In [ ]:
# -*- coding: utf-8 -*-
"""
SMPC — EV BUFFER MODEL (PAPER-ALIGNED REVISION)
-------------------------------------------------------------------
This revision aligns the controller objective with the manuscript:

1) EV regularization (Eq. 16) is now a PER-BUS QUADRATIC ramp + peak penalty:
       lambda_dP * (P_k^{EV,i} - P_{k-1}^{EV,i})^2  +  lambda_pk * (P_k^{EV,i})^2
   (previously: L1 ramp + quadratic peak on the FLEET TOTAL). The L1 auxiliary
   variables dP_ev_pos / dP_ev_neg have been removed.
2) Battery degradation (Eq. 13) penalizes DISCHARGE ONLY: rho_deg * P_B_dc
   (previously: rho_deg * (P_B_ch + P_B_dc)).
3) Terminal cost (Eq. 14) is now ONE-SIDED: rho_term * max(E_ref - E_N, 0)^2,
   implemented with a non-negative slack term_short >= E_ref - E_N
   (previously: two-sided quadratic).
4) EV lower-bound slack penalty RHO_EV_SLACK = 1000 (Table I lists rho^{EV}_viol = 50;
   reverted to 1000 per the configuration used for the reported runs).
5) Generator efficiency unified at 0.35. The diesel cost c_gas is already the
   per-electric-kWh value (DIESEL_PRICE / (kWh_th_per_L * eta)), so the objective
   does NOT divide by eta again; eta_g is retained only for documentation.
6) CTMC transition probabilities (Eq. 10) renamed/clarified and now use dt:
       p_{ON->OFF} = 1 - exp(-psi * dt),  p_{OFF->ON} = 1 - exp(-nu * dt).
   With psi=0.10/h, nu=0.50/h: steady-state availability = nu/(psi+nu) = 0.833,
   mean outage duration = 1/nu = 2 h.

7) GP LOAD FORECASTING (Section IV-A) now matches the manuscript text:
   - THREE-MONTH SLIDING WINDOW. The GP is CONDITIONED ("trained") on the second
     and third preceding months (M-2, M-3) relative to the forecast month M.
     Kernel hyperparameters are SELECTED by maximizing the held-out log predictive
     likelihood on the MOST RECENT preceding month (M-1), which is excluded from
     conditioning. (scikit-learn has no native train/validation split, so this is
     an explicit search: for each candidate theta, condition a GP on M-2,M-3 and
     score the Gaussian log-likelihood of M-1; keep the best theta.)
   - JOINT POSTERIOR SAMPLING. Load-trajectory scenarios are drawn with
     GaussianProcessRegressor.sample_y, i.e. from the joint posterior covariance,
     so they carry the temporal correlation encoded by the kernel (previously the
     scenarios were independent per-hour Gaussian perturbations of the mean).
   - The weekday/weekend feature now also flags PUBLIC HOLIDAYS (Ghana) when the
     optional 'holidays' package is installed (weekend-only fallback otherwise).
   This requires a dataset extending before January 2023 so the first forecast
   month has its full M-1/M-2/M-3 history. Use:
     synthetic_ashesi_2022-09_to_2023-12_hourly_filled_pv.csv

NOTE on dissipation averaging (Eq. 11): P_DISS_AVG_MODE stays
"full_unplugged_interval". The loss term is applied at every unplugged step
(1 - z), so spreading route energy over the whole unplugged interval integrates
to exactly the trip energy. Dividing by drive time only would over-count energy.
This is a manuscript-text reconciliation (Eq. 11 wording), not a code change.

Requirements
------------
pip install gurobipy numpy pandas scikit-learn scipy holidays
('holidays' is optional; without it the weekend/holiday feature uses weekends only.)
"""

import os
import math
import time
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ExpSineSquared, WhiteKernel, ConstantKernel as C
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize


# =========================================================
# SETTINGS
# =========================================================
ALLOW_BESS_DISCHARGE = True

# SMPC scenario settings
M_SCEN = 30
NOISE_SCALE = 1.0
RNG_SEED = 42

# ---- GP load-forecasting (paper-aligned three-month sliding window) ----
USE_HOLIDAYS = True            # include public holidays in the weekend/holiday feature (needs 'holidays' pkg)
HOLIDAY_COUNTRY = "GH"         # Ghana
GP_OPT_TRAIN_CAP = 600         # max points used to CONDITION the GP during hyperparameter search (M-2,M-3)
GP_OPT_VAL_CAP = 500           # max points scored on the held-out month M-1 during the search
GP_FINAL_TRAIN_CAP = 1500      # max points used to condition the FINAL forecasting GP (M-2,M-3)
GP_N_RESTARTS = 2              # random restarts for the hyperparameter search (plus the default start)
_GP_JITTER = 1e-6              # numerical jitter added to the GP diagonal (stability)

# Outage CTMC scenario settings (psi = ON->OFF outage rate, nu = OFF->ON restoration rate)
OUT_LAMBDA = 0.10  # psi: ON->OFF per hour
OUT_MU = 0.50      # nu : OFF->ON per hour

# EV settings
NUM_BUSES = 6
P_PORT_KW = 60.0
EV_BATT_KWH = 300.0
EV_MIN_KWH = 10.0
EV_CHG_EFF = 0.95
EV_INIT_KWH_TOTAL = 200.0  # retained only as legacy aggregate reference

# Scientifically interpretable first-pass alpha sweep
ALPHA_SWEEP = [0.00, 0.10, 0.20, 0.30, 0.40]
APRIL_ONLY = "2023-04"
MONTHS_TO_RUN = ["2023-01", "2023-04", "2023-06", "2023-07"]
CONTROLLER_NAME = "SMPC"
WARM_START_DAYS = 4
# Real December 2022 now exists in the extended dataset, so the January warm-start
# no longer needs to wrap onto December 2023. (Moot for the reported months, which
# use MONTHLY_BUS_E0_OVERRIDE, but kept correct in case the override is removed.)
JANUARY_WARM_START_USE_DECEMBER_2023 = False

# Route-loss averaging semantics for P_i^{diss,avg}
# "full_unplugged_interval" is the default because the loss is applied at every
# unplugged step (1 - z); spreading route energy over the whole unplugged interval
# integrates to exactly the trip energy. Eq. 11's text reads as driving-time power,
# but driving_time_only would over-count when applied over all unplugged steps.
P_DISS_AVG_MODE = "full_unplugged_interval"

# Optional measured initial bus energy override.
BUS_E0_OVERRIDE = None

# Fixed month-specific initial bus energies aligned to the SMPC study.
MONTHLY_BUS_E0_OVERRIDE = {
    "2023-01": np.array([17.002, 26.294, 50.328, 15.422, 11.306, 24.373], dtype=float),
    "2023-04": np.array([42.102, 35.197, 53.167, 32.667, 30.64, 30.809], dtype=float),
    "2023-06": np.array([44.128, 36.63, 52.471, 34.019, 33.232, 30.881], dtype=float),
    "2023-07": np.array([47.896, 37.095, 52.286, 35.034, 32.824, 35.349], dtype=float),
}

HOLD_COMMON_INITIAL_BUS_ENERGY_ACROSS_ALPHA = True
COMMON_INITIAL_BUS_ENERGY_REFERENCE_ALPHA = 0.00

FIX_CHARGER_CAPACITY_TO_NOMINAL = True

# Warm-start settings for initialization when measured E0 is unavailable.
WARM_START_ENABLED = True
WARM_START_START = "2023-03-28 00:00:00"
WARM_START_SEED_MODE = "route_aware_mid"

# Deterministic debug run
RUN_DETERMINISTIC_DEBUG = False
DEBUG_ALPHA = 0.00
DEBUG_WINDOW_START = "2023-04-01 00:00:00"
DEBUG_WINDOW_END = "2023-04-07 23:00:00"

FLOOR_CLIP_WARN_TOL_KWH = 1e-3

# EV penalty / regularization
RHO_EV_SLACK = 1000.0        # EV lower-bound slack penalty (reverted to 1000)
RHO_EV_FLOOR_CLIP = 5000.0
LAMBDA_EV_RAMP_Q = 5.0       # quadratic per-bus ramp weight (Eq. 16, lambda_dP)
LAMBDA_EV_PEAK_Q = 0.02      # quadratic per-bus peak weight (Eq. 16, lambda_pk)

# EV treatment
USE_DETERMINISTIC_EV_SCENARIO = False
USE_STRESS_RANDOM_SCENARIO = True
PLAN_WITH_ACTUAL_EV_FULL_HORIZON = False

# Stochastic depot-charging realization knobs
MIDDAY_ARR_STD_H = 0.60
EVENING_ARR_STD_H = 0.75
MIDDAY_DUR_DISC = np.array([-1, 0, 1, 2], dtype=int)
MIDDAY_DUR_PROB = np.array([0.15, 0.45, 0.25, 0.15], dtype=float)
EVENING_DUR_DISC = np.array([-1, 0, 1, 2], dtype=int)
EVENING_DUR_PROB = np.array([0.10, 0.35, 0.35, 0.20], dtype=float)
MIDDAY_DUR_CONT_STD_H = 0.50
EVENING_DUR_CONT_STD_H = 0.75
MIDDAY_DEMAND_MULT_STD = 0.20
EVENING_DEMAND_MULT_STD = 0.18
OVERNIGHT_DEMAND_MULT_STD = 0.05
DAY_DERATE_LOW = 0.85
DAY_DERATE_HIGH = 1.00
SESSION_POWER_MULT_STD = 0.08
SESSION_POWER_MULT_LOW = 0.70
SESSION_POWER_MULT_HIGH = 1.15
DEMAND_MULT_LOW = 0.55
DEMAND_MULT_HIGH = 1.45

# Route energy ingredients
EV_PER_KM_KWH = 1.20
EV_AUX_FRAC = 0.10
DIST_KM = {
    "CTK": 33.0, "Haatso/KFC": 17.0, "Lapaz": 27.0, "Spintex": 34.0,
    "BerekusoTownship": 3.0, "Atomic": 15.0, "Tieman/Engine": 10.0,
    "Aboum": 8.0, "Shiashie-37-CTK": 33.0,
}

# Electricity / diesel tariff settings
FLAT_ELECTRICITY_PRICE_GHS_PER_KWH_ELEC = 2.5

DIESEL_PRICE_GHS_PER_L = 14.0
KWH_TH_PER_L_DIESEL = 9.7
ETA_GEN_ELEC = 0.35  # single generator efficiency used to derive the per-electric diesel cost
# c_gas is the per-ELECTRIC-kWh diesel cost (= 14 / (9.7 * 0.35) = 4.124 GHS/kWh_elec),
# matching Table I c^d = 4.12. The objective therefore uses c_gas * P_g directly and
# does NOT divide by eta again (avoids the double-counting implied by Eq. 13 + Table I).
C_DIESEL_GHS_PER_KWH_ELEC = DIESEL_PRICE_GHS_PER_L / (KWH_TH_PER_L_DIESEL * ETA_GEN_ELEC)

# Diesel cost / emissions settings
KWH_TH_PER_LITRE_DIESEL = KWH_TH_PER_L_DIESEL
KGCO2_PER_LITRE_DIESEL = 2.69
DIESEL_MG_EF_KGCO2_PER_KWH = 0.8


# =========================================================
# HELPERS
# =========================================================
def find_path(fname: str) -> str:
    if os.path.exists(fname):
        return fname
    alt = os.path.join("/mnt/data", fname)
    return alt if os.path.exists(alt) else fname


def get_month_specific_bus_e0_override(target_start: pd.Timestamp):
    mon = pd.Timestamp(target_start).strftime("%Y-%m")
    arr = MONTHLY_BUS_E0_OVERRIDE.get(mon, None)
    if arr is None:
        return None
    arr = np.asarray(arr, float).reshape(-1)
    if arr.size != NUM_BUSES:
        raise ValueError(f"MONTHLY_BUS_E0_OVERRIDE[{mon!r}] must have length {NUM_BUSES}.")
    return arr


def make_flat_electricity_prices(index, flat_price=FLAT_ELECTRICITY_PRICE_GHS_PER_KWH_ELEC):
    return np.full(len(index), float(flat_price), dtype=float)


def make_flat_diesel_prices(index, flat_price=C_DIESEL_GHS_PER_KWH_ELEC):
    return np.full(len(index), float(flat_price), dtype=float)


def diesel_co2_metrics(P_g_kW, dt_h, ef_kgco2_per_kwh=DIESEL_MG_EF_KGCO2_PER_KWH):
    P_g_kW = np.asarray(P_g_kW, float)
    E_diesel_e_kWh = float(np.sum(P_g_kW * float(dt_h)))
    co2_kg = E_diesel_e_kWh * float(ef_kgco2_per_kwh)
    litres_implied = co2_kg / float(KGCO2_PER_LITRE_DIESEL)
    return dict(
        Diesel_Elec_MWh=E_diesel_e_kWh / 1000.0,
        Diesel_Litres_implied=litres_implied,
        CO2_tonnes=co2_kg / 1000.0,
        Emissions_Factor_kgCO2_per_kWh=float(ef_kgco2_per_kwh),
    )


def diesel_co2_timeseries(P_g_kW, dt_h, ef_kgco2_per_kwh=DIESEL_MG_EF_KGCO2_PER_KWH):
    P_g_kW = np.asarray(P_g_kW, float)
    E_step_kWh = P_g_kW * float(dt_h)
    co2_kg_step = E_step_kWh * float(ef_kgco2_per_kwh)
    litres_step = co2_kg_step / float(KGCO2_PER_LITRE_DIESEL)
    return {
        "diesel_elec_kWh_step": E_step_kWh,
        "diesel_co2_kg_step": co2_kg_step,
        "diesel_co2_tonnes_step": co2_kg_step / 1000.0,
        "diesel_litres_implied_step": litres_step,
    }


def compute_busnode_metrics(shortfall_bus_nodes: np.ndarray):
    x = np.asarray(shortfall_bus_nodes, float)
    flat = x.reshape(-1)
    N_nodes = int(flat.size)
    viol = flat[flat > 1e-6]
    N_viol = int(viol.size)
    frac = float(N_viol) / float(N_nodes) if N_nodes > 0 else 0.0
    return dict(
        nodes_total=N_nodes,
        nodes_viol=N_viol,
        frac_viol=frac,
        avg_short_kWh=float(viol.mean()) if N_viol > 0 else 0.0,
        max_short_kWh=float(viol.max()) if N_viol > 0 else 0.0,
        total_short_kWh=float(viol.sum()) if N_viol > 0 else 0.0,
    )


def _clip_round_hour(x, lo, hi):
    return int(np.clip(np.round(x), lo, hi))


def _e_for_km(d_km):
    return d_km * EV_PER_KM_KWH * (1.0 + EV_AUX_FRAC)


def _route_propulsion_energy(route_key, round_trip):
    if route_key not in DIST_KM:
        return 0.0
    km = DIST_KM[route_key] * (2 if round_trip else 1)
    return _e_for_km(km)


def _ceil_hour(h, m):
    return int(math.ceil(h + m / 60.0 - 1e-9))


def _classify_session(a, d, is_final=False):
    if a == 0 and d <= 6:
        return "overnight"
    if is_final:
        return "evening"
    if d <= 17:
        return "midday"
    return "evening"


# =========================================================
# EV schedule generation
# =========================================================

def generate_ashesi_deterministic_day(N=24, dt=1.0, p_port=P_PORT_KW):
    p_bus = np.full(NUM_BUSES, p_port, float)

    MORNING_ARR = {
        0: _ceil_hour(7, 50), 1: _ceil_hour(7, 50), 2: _ceil_hour(8, 30),
        3: _ceil_hour(8, 5), 4: _ceil_hour(8, 35), 5: _ceil_hour(7, 35),
    }

    aft_duties = [
        (0, 13, 30, "Atomic", 90, True),
        (1, 13, 30, "Tieman/Engine", 70, True),
        (2, 15, 0, "Aboum", 50, True),
    ]
    eve_duties = [
        (3, 17, 10, "Lapaz", 80, False),
        (4, 17, 10, "Spintex", 130, False),
        (0, 17, 10, "Shiashie-37-CTK", 90, False),
        (1, 17, 10, "Haatso/KFC", 80, False),
        (5, 18, 30, "Aboum", 70, True),
        (5, 20, 0, "Aboum", 50, True),
    ]
    duties = aft_duties + eve_duties
    duties.sort(key=lambda x: (x[0], x[1] * 60 + x[2]))

    def session_charge_target(route_key, round_trip):
        km = DIST_KM[route_key] * (2 if round_trip else 1)
        return 1.10 * _e_for_km(km) + 0.10 * EV_BATT_KWH

    sessions = []
    sid = 0

    def add_session(bus, a, d, e_kWh, session_class, route_key="", round_trip=False, route_drive_h=0.0):
        nonlocal sid
        a = int(max(0, a))
        d = int(max(0, d))
        if d > a and e_kWh >= 0:
            route_energy_kWh = _route_propulsion_energy(route_key, round_trip) if route_key in DIST_KM else 0.0
            sessions.append(
                (
                    bus, sid, a, d, float(e_kWh), float(p_bus[bus]), session_class, route_key,
                    int(round_trip), float(route_energy_kWh), float(route_drive_h)
                )
            )
            sid += 1

    for b in range(NUM_BUSES):
        add_session(
            b, a=0, d=6, e_kWh=0.25 * EV_BATT_KWH,
            session_class="overnight", route_key="overnight", round_trip=False, route_drive_h=0.0
        )

    for b in range(NUM_BUSES):
        arrive = MORNING_ARR[b]
        my = [d for d in duties if d[0] == b]
        for (_, hh, mm, route_key, dur_min, round_trip) in my:
            dep_node = _ceil_hour(hh, mm)
            e_need = session_charge_target(route_key, round_trip=round_trip)
            sess_class = _classify_session(arrive, dep_node, is_final=False)
            add_session(
                b, a=arrive, d=dep_node, e_kWh=e_need,
                session_class=sess_class, route_key=route_key, round_trip=round_trip,
                route_drive_h=float(dur_min) / 60.0
            )
            if round_trip:
                ret_node = _ceil_hour(hh, mm + dur_min)
                arrive = max(arrive, ret_node)
            else:
                arrive = N
        if arrive < N:
            add_session(
                b, a=max(arrive, 21), d=N, e_kWh=0.10 * EV_BATT_KWH,
                session_class="evening", route_key="final_topup", round_trip=False, route_drive_h=0.0
            )

    cols = [
        "bus", "sess", "a", "d", "e_kWh", "p_bus_kW",
        "session_class", "route_key", "round_trip_flag", "route_energy_kWh", "route_drive_h"
    ]
    if sessions:
        sessions_df = pd.DataFrame(sessions, columns=cols).sort_values(["bus", "a", "d"]).reset_index(drop=True)
    else:
        sessions_df = pd.DataFrame(columns=cols)

    info = {"P_EV_max": float(np.sum(p_bus)), "sessions": sessions_df.copy()}
    return sessions_df, info


def generate_ashesi_schedule_based_stochastic_day(N=24, dt=1.0, base_port=P_PORT_KW, rng_seed=RNG_SEED):
    rng = np.random.default_rng(rng_seed)
    base_df, _ = generate_ashesi_deterministic_day(N=N, dt=dt, p_port=base_port)
    if FIX_CHARGER_CAPACITY_TO_NOMINAL:
        per_bus_day_derate = np.ones(NUM_BUSES, dtype=float)
    else:
        per_bus_day_derate = rng.uniform(DAY_DERATE_LOW, DAY_DERATE_HIGH, size=NUM_BUSES)

    sessions = []
    sid = 0

    def add_realized_session(bus, a, d, d_deadline, e_kWh, p_bus_kW,
                             session_class, route_key, round_trip_flag,
                             route_energy_kWh, route_drive_h, realized_flag=1):
        nonlocal sid
        a = int(np.clip(a, 0, N))
        d = int(np.clip(d, 0, N))
        d_deadline = int(np.clip(d_deadline, 0, N))
        if d <= a or p_bus_kW <= 1e-9:
            return
        sessions.append(
            (
                bus, sid, a, d, d_deadline, float(e_kWh), float(p_bus_kW), session_class,
                route_key, int(round_trip_flag), float(route_energy_kWh), float(route_drive_h), int(realized_flag)
            )
        )
        sid += 1

    for _, row in base_df.iterrows():
        bus = int(row["bus"])
        a0 = int(row["a"])
        d0 = int(row["d"])
        e0 = float(row["e_kWh"])
        p0 = float(row["p_bus_kW"])
        sess_class = str(row["session_class"])
        route_key = str(row["route_key"])
        round_trip_flag = int(row["round_trip_flag"])
        route_energy_nom = float(row.get("route_energy_kWh", 0.0))
        route_drive_h = float(row.get("route_drive_h", 0.0))

        if FIX_CHARGER_CAPACITY_TO_NOMINAL:
            day_derate = 1.0
            power_mult = 1.0
            p_eff = p0
        else:
            day_derate = float(per_bus_day_derate[bus])
            power_mult = float(np.clip(rng.normal(1.0, SESSION_POWER_MULT_STD),
                                       SESSION_POWER_MULT_LOW, SESSION_POWER_MULT_HIGH))
            p_eff = p0 * day_derate * power_mult
        d_deadline = d0

        if sess_class == "overnight":
            a = a0
            d = d0
            e_mult = float(np.clip(rng.normal(1.0, OVERNIGHT_DEMAND_MULT_STD), 0.90, 1.10))
            e_real = max(1.0, e0 * e_mult)
            route_energy_real = 0.0
        else:
            arr_std = MIDDAY_ARR_STD_H if sess_class == "midday" else EVENING_ARR_STD_H
            a = _clip_round_hour(a0 + rng.normal(0.0, arr_std), 0, d_deadline)
            base_dur = max(1, d0 - a0)
            if sess_class == "midday":
                dur_disc = int(rng.choice(MIDDAY_DUR_DISC, p=MIDDAY_DUR_PROB))
                dur_cont = float(rng.normal(0.0, MIDDAY_DUR_CONT_STD_H))
                demand_std = MIDDAY_DEMAND_MULT_STD
            else:
                dur_disc = int(rng.choice(EVENING_DUR_DISC, p=EVENING_DUR_PROB))
                dur_cont = float(rng.normal(0.0, EVENING_DUR_CONT_STD_H))
                demand_std = EVENING_DEMAND_MULT_STD
            dur_real = int(max(0, np.round(base_dur + dur_disc + dur_cont)))
            d = int(min(d_deadline, a + dur_real))
            e_mult = float(np.clip(rng.normal(1.0, demand_std), DEMAND_MULT_LOW, DEMAND_MULT_HIGH))
            e_real = max(1.0, e0 * e_mult)
            route_energy_real = max(0.0, route_energy_nom * e_mult)

        add_realized_session(
            bus, a, d, d_deadline, e_real, p_eff, sess_class, route_key, round_trip_flag,
            route_energy_real, route_drive_h, realized_flag=1
        )

    cols = [
        "bus", "sess", "a", "d", "d_deadline", "e_kWh", "p_bus_kW",
        "session_class", "route_key", "round_trip_flag", "route_energy_kWh", "route_drive_h", "realized_flag"
    ]
    if sessions:
        sessions_df = pd.DataFrame(sessions, columns=cols).sort_values(["bus", "d_deadline", "a", "d"]).reset_index(drop=True)
    else:
        sessions_df = pd.DataFrame(columns=cols)
    return sessions_df


def build_month_sessions_deterministic(index_span: pd.DatetimeIndex):
    days = pd.DatetimeIndex(index_span.normalize()).unique()
    sessions_all = []
    P_EV_max = None
    for day in days:
        sess_df, info = generate_ashesi_deterministic_day(N=24, dt=1.0, p_port=P_PORT_KW)
        if P_EV_max is None:
            P_EV_max = float(info["P_EV_max"])
        mask = (index_span.normalize() == day)
        idx = np.where(mask)[0]
        if len(idx) == 0:
            continue
        t0 = int(idx[0])
        tmp = sess_df.copy()
        tmp["a"] += t0
        tmp["d"] += t0
        sessions_all.append(tmp)
    cols = [
        "bus", "sess", "a", "d", "e_kWh", "p_bus_kW",
        "session_class", "route_key", "round_trip_flag", "route_energy_kWh", "route_drive_h"
    ]
    if sessions_all:
        sessions_all = pd.concat(sessions_all, ignore_index=True)
    else:
        sessions_all = pd.DataFrame(columns=cols)
    return sessions_all, float(P_EV_max)


def build_month_sessions_stochastic(index_span: pd.DatetimeIndex, rng_seed=RNG_SEED):
    days = pd.DatetimeIndex(index_span.normalize()).unique()
    sessions_all = []
    for day_idx, day in enumerate(days):
        sess_df = generate_ashesi_schedule_based_stochastic_day(
            N=24, dt=1.0, base_port=P_PORT_KW, rng_seed=rng_seed + day_idx
        )
        mask = (index_span.normalize() == day)
        idx = np.where(mask)[0]
        if len(idx) == 0:
            continue
        t0 = int(idx[0])
        tmp = sess_df.copy()
        tmp["a"] += t0
        tmp["d"] += t0
        if "d_deadline" in tmp.columns:
            tmp["d_deadline"] += t0
        sessions_all.append(tmp)
    cols = [
        "bus", "sess", "a", "d", "d_deadline", "e_kWh", "p_bus_kW",
        "session_class", "route_key", "round_trip_flag", "route_energy_kWh", "route_drive_h", "realized_flag"
    ]
    if sessions_all:
        sessions_all = pd.concat(sessions_all, ignore_index=True)
    else:
        sessions_all = pd.DataFrame(columns=cols)
    return sessions_all, float(NUM_BUSES * P_PORT_KW)


def build_bus_availability_arrays(index_span: pd.DatetimeIndex, sessions_all: pd.DataFrame, n_buses=NUM_BUSES):
    T = len(index_span)
    z = np.zeros((n_buses, T), dtype=float)
    pmax = np.zeros((n_buses, T), dtype=float)
    if sessions_all is None or len(sessions_all) == 0:
        return z, pmax
    for _, row in sessions_all.iterrows():
        b = int(row["bus"])
        if not (0 <= b < n_buses):
            continue
        a = int(row["a"])
        d = int(row["d"])
        p = float(row["p_bus_kW"])
        a = max(0, min(T, a))
        d = max(0, min(T, d))
        if d > a:
            z[b, a:d] = 1.0
            pmax[b, a:d] = np.maximum(pmax[b, a:d], p)
    return z, pmax


def build_departure_requirement_arrays(index_span: pd.DatetimeIndex, sessions_all: pd.DataFrame, n_buses=NUM_BUSES):
    T = len(index_span)
    req = np.zeros((n_buses, T), dtype=float)
    event_count = np.zeros((n_buses, T), dtype=int)
    if sessions_all is None or len(sessions_all) == 0:
        return req, event_count
    dep_col = "d_deadline" if "d_deadline" in sessions_all.columns else "d"
    for _, row in sessions_all.iterrows():
        route_energy = float(row.get("route_energy_kWh", 0.0))
        if route_energy <= 1e-9:
            continue
        b = int(row["bus"])
        dep = int(row[dep_col])
        if 0 <= b < n_buses and 0 <= dep < T:
            req[b, dep] = max(req[b, dep], EV_MIN_KWH + route_energy)
            event_count[b, dep] += 1
    return req, event_count


def compute_bus_avg_dissipation_from_daily_sessions(day_sessions: pd.DataFrame,
                                                    N=24,
                                                    n_buses=NUM_BUSES,
                                                    averaging_mode=P_DISS_AVG_MODE):
    """
    Compute P_i^{diss,avg} from ROUTE ENERGY ONLY (Eq. 11). With
    "full_unplugged_interval" the per-hour rate integrated over the unplugged
    interval equals exactly the trip energy, conserving energy.
    """
    p_diss = np.zeros(n_buses, dtype=float)
    if day_sessions is None or len(day_sessions) == 0:
        return p_diss

    valid_modes = {"full_unplugged_interval", "driving_time_only"}
    if averaging_mode not in valid_modes:
        raise ValueError(f"Unknown averaging_mode={averaging_mode}. Valid options: {sorted(valid_modes)}")

    for b in range(n_buses):
        s = day_sessions.loc[day_sessions["bus"].astype(int) == b].copy().sort_values(["a", "d"]).reset_index(drop=True)
        if len(s) == 0:
            continue

        total_route_energy = 0.0
        total_avg_hours = 0.0
        for j in range(len(s)):
            route_energy = float(s.loc[j, "route_energy_kWh"]) if "route_energy_kWh" in s.columns else 0.0
            if route_energy <= 1e-9:
                continue

            d_j = int(s.loc[j, "d"])
            if j < len(s) - 1:
                a_next = int(s.loc[j + 1, "a"])
            else:
                a_next = int(s.loc[0, "a"]) + N

            away_dur = max(a_next - d_j, 0)
            drive_h = float(s.loc[j, "route_drive_h"]) if "route_drive_h" in s.columns else 0.0

            if averaging_mode == "full_unplugged_interval":
                denom_h = float(away_dur)
            else:
                denom_h = float(max(drive_h, 1e-9))

            if denom_h > 1e-9:
                total_route_energy += route_energy
                total_avg_hours += denom_h

        p_diss[b] = total_route_energy / total_avg_hours if total_avg_hours > 1e-9 else 0.0
    return p_diss


def compute_departure_violation_metrics(E_bus_nodes: np.ndarray,
                                        dep_req_by_bus_time: np.ndarray,
                                        dep_event_count_by_bus_time: np.ndarray):
    E = np.asarray(E_bus_nodes, float)
    req = np.asarray(dep_req_by_bus_time, float)
    counts = np.asarray(dep_event_count_by_bus_time, int)

    T_nodes = E.shape[0]
    T_req = req.shape[1]
    T = min(T_nodes, T_req)
    gap = np.maximum(req[:, :T].T - E[:T, :], 0.0)
    active = (counts[:, :T].T > 0)

    viol_vals = gap[active]
    n_events = int(active.sum())
    n_viol = int(np.sum(viol_vals > 1e-6))
    return {
        "events_total": n_events,
        "events_viol": n_viol,
        "viol_frac": float(n_viol / n_events) if n_events > 0 else 0.0,
        "avg_short_kWh": float(viol_vals[viol_vals > 1e-6].mean()) if n_viol > 0 else 0.0,
        "max_short_kWh": float(viol_vals.max()) if n_events > 0 else 0.0,
        "total_short_kWh": float(np.sum(viol_vals)),
        "gap_by_time_kWh": gap.sum(axis=1),
        "viol_events_by_time": np.sum((gap > 1e-6) & active, axis=1).astype(int),
        "active_events_by_time": np.sum(active, axis=1).astype(int),
    }


def compute_session_violation_metrics(shortfall_bus_nodes: np.ndarray,
                                      sessions_all: pd.DataFrame,
                                      T_window: int,
                                      n_buses: int = NUM_BUSES):
    x = np.asarray(shortfall_bus_nodes, float)
    if x.ndim != 2:
        raise ValueError("shortfall_bus_nodes must be a 2D node-by-bus array.")
    if sessions_all is None or len(sessions_all) == 0:
        return {"sessions_total": 0, "sessions_viol": 0, "viol_rate": 0.0}

    T_eff = min(int(T_window), int(x.shape[0] - 1))
    n_total = 0
    n_viol = 0

    for _, row in sessions_all.iterrows():
        b = int(row["bus"])
        if not (0 <= b < n_buses):
            continue
        a = int(row["a"])
        d = int(row["d"])

        if d <= 0 or a >= T_eff:
            continue
        a0 = max(0, a)
        d0 = min(T_eff, d)
        if d0 <= a0:
            continue

        n_total += 1
        if np.any(x[a0:d0 + 1, b] > 1e-6):
            n_viol += 1

    return {
        "sessions_total": int(n_total),
        "sessions_viol": int(n_viol),
        "viol_rate": float(n_viol / n_total) if n_total > 0 else 0.0,
    }


def build_route_aware_warmstart_seed(day_sessions: pd.DataFrame, n_buses=NUM_BUSES):
    seed = np.full(n_buses, 0.50 * EV_BATT_KWH, dtype=float)
    if day_sessions is None or len(day_sessions) == 0:
        return seed
    for b in range(n_buses):
        s = day_sessions.loc[day_sessions["bus"].astype(int) == b].copy()
        route_e = s.get("route_energy_kWh", pd.Series(dtype=float)).astype(float)
        max_route = float(route_e.max()) if len(route_e) > 0 else 0.0
        daily_route = float(route_e.sum()) if len(route_e) > 0 else 0.0
        seed[b] = np.clip(
            max(0.50 * EV_BATT_KWH, EV_MIN_KWH + 1.25 * max_route, 0.20 * EV_BATT_KWH + 0.15 * daily_route),
            EV_MIN_KWH,
            0.85 * EV_BATT_KWH
        )
    return seed


def build_ev_schedules_for_span(index_span: pd.DatetimeIndex, deterministic_actual=False, rng_seed=RNG_SEED):
    nominal_sessions_full, P_EV_max = build_month_sessions_deterministic(index_span)
    z_nom_full, pmax_nom_full = build_bus_availability_arrays(index_span, nominal_sessions_full, NUM_BUSES)

    if deterministic_actual:
        actual_sessions_full = nominal_sessions_full.copy()
        scenario_label = "Deterministic timetable (nominal = actual)"
    else:
        actual_sessions_full, _ = build_month_sessions_stochastic(index_span, rng_seed=rng_seed)
        if FIX_CHARGER_CAPACITY_TO_NOMINAL:
            scenario_label = "Stochastic EV realization with fixed nominal charger capacity"
        else:
            scenario_label = "Stochastic EV realization used in both planning and execution"

    z_actual_full, pmax_actual_full = build_bus_availability_arrays(index_span, actual_sessions_full, NUM_BUSES)
    dep_req_actual_full, dep_event_count_full = build_departure_requirement_arrays(index_span, actual_sessions_full, NUM_BUSES)

    return {
        "nominal_sessions_full": nominal_sessions_full,
        "actual_sessions_full": actual_sessions_full,
        "z_nom_full": z_nom_full,
        "pmax_nom_full": pmax_nom_full,
        "z_actual_full": z_actual_full,
        "pmax_actual_full": pmax_actual_full,
        "dep_req_actual_full": dep_req_actual_full,
        "dep_event_count_full": dep_event_count_full,
        "P_EV_max": float(P_EV_max),
        "scenario_label": scenario_label,
    }


def build_time_varying_bus_dissipation_profile(index_span: pd.DatetimeIndex,
                                              sessions_all: pd.DataFrame,
                                              n_buses: int = NUM_BUSES,
                                              averaging_mode: str = P_DISS_AVG_MODE):
    T = len(index_span)
    prof = np.zeros((n_buses, T), dtype=float)
    if sessions_all is None or len(sessions_all) == 0:
        return prof

    norm = pd.DatetimeIndex(index_span.normalize())
    for day in norm.unique():
        idx = np.where(norm == day)[0]
        if len(idx) == 0:
            continue
        t0 = int(idx[0])
        day_sess = sessions_all.loc[
            (sessions_all["a"].astype(int) >= t0) &
            (sessions_all["a"].astype(int) < t0 + 24)
        ].copy()
        p_day = compute_bus_avg_dissipation_from_daily_sessions(
            day_sess, N=24, n_buses=n_buses, averaging_mode=averaging_mode
        )
        prof[:, idx] = np.asarray(p_day, float).reshape(-1, 1)
    return prof


def prepare_window_inputs(df: pd.DataFrame, window_start: pd.Timestamp, window_end: pd.Timestamp, H_LOOK: int):
    index_window = df.loc[window_start:window_end].index
    if len(index_window) == 0:
        raise ValueError(f"No data found in window [{window_start}, {window_end}]")
    index_span = pd.date_range(start=index_window[0], periods=len(index_window) + H_LOOK, freq="H")
    dfe = df.reindex(index_span).copy()
    if "PV_P_ac_KW" in dfe.columns:
        dfe = dfe.rename(columns={"PV_P_ac_KW": "PV_P_ac_kW"})
    req = ["Load_kW", "PV_P_ac_kW", "Grid_Cap_kW"]
    if dfe[req].isna().any().any():
        missing = dfe[req].isna().sum().to_dict()
        raise ValueError(f"Dataset has NaNs in window {window_start}..{window_end} for {req}: {missing}")

    return {
        "index_window": index_window,
        "index_span": index_span,
        "dfe": dfe,
        "P_pv_max_full": np.maximum(0.0, dfe["PV_P_ac_kW"].to_numpy(float)),
        "P_grid_cap_full": dfe["Grid_Cap_kW"].to_numpy(float),
        "c_elec_full": make_flat_electricity_prices(index_span),
        "c_gas_full": make_flat_diesel_prices(index_span),
    }


def extend_df_with_year_wrap_hours(df: pd.DataFrame, extra_hours: int) -> pd.DataFrame:
    extra_hours = int(max(0, extra_hours))
    if extra_hours == 0:
        return df.copy()

    df_ext = df.copy()
    start_src = pd.Timestamp(df.index.min())
    end_src = start_src + pd.Timedelta(hours=extra_hours - 1)
    extra = df.loc[start_src:end_src].copy()
    if len(extra) < extra_hours:
        raise ValueError(f"Not enough source hours to wrap {extra_hours} hour(s) from the start of the dataset.")
    shift = (pd.Timestamp(df.index.max()) + pd.Timedelta(hours=1)) - start_src
    extra.index = extra.index + shift
    df_ext = pd.concat([df_ext, extra])
    return df_ext.sort_index()


def get_warm_start_bounds_for_target(target_start: pd.Timestamp, lookback_days: int = WARM_START_DAYS):
    target_start = pd.Timestamp(target_start)
    lookback_days = int(max(1, lookback_days))

    if JANUARY_WARM_START_USE_DECEMBER_2023 and target_start == pd.Timestamp("2023-01-01 00:00:00"):
        warm_end = pd.Timestamp("2023-12-31 23:00:00")
        warm_start = warm_end - pd.Timedelta(days=lookback_days) + pd.Timedelta(hours=1)
        return warm_start, warm_end, "dec_2023_wrap"

    warm_end = target_start - pd.Timedelta(hours=1)
    warm_start = target_start - pd.Timedelta(days=lookback_days)
    return warm_start, warm_end, "previous_days"


def compute_initial_bus_energy_for_window(df: pd.DataFrame,
                                          target_start: pd.Timestamp,
                                          H_LOOK: int,
                                          alpha_buffer: float,
                                          p_diss_avg: np.ndarray,
                                          nominal_day_sessions: pd.DataFrame,
                                          E_B_min: float,
                                          E_B_max: float,
                                          P_B_max: float,
                                          eta_ch: float,
                                          eta_dc: float,
                                          eta_g: float,
                                          rho_deg: float,
                                          rho_curt: float,
                                          rho_shed: float):
    month_override = get_month_specific_bus_e0_override(target_start)
    if month_override is not None:
        return month_override, "monthly_fixed_override", 0.0, None

    if BUS_E0_OVERRIDE is not None:
        E_bus0_init = np.asarray(BUS_E0_OVERRIDE, float).reshape(-1)
        if E_bus0_init.size != NUM_BUSES:
            raise ValueError("BUS_E0_OVERRIDE must have length NUM_BUSES.")
        return E_bus0_init, "global_measured_override", 0.0, None

    if WARM_START_ENABLED:
        target_start = pd.Timestamp(target_start)
        warm_start, warm_end, warm_tag = get_warm_start_bounds_for_target(target_start, lookback_days=WARM_START_DAYS)
        df_for_warm = extend_df_with_year_wrap_hours(df, H_LOOK) if warm_tag == "dec_2023_wrap" else df

        if warm_end >= warm_start and warm_start >= df_for_warm.index.min() and warm_end <= df_for_warm.index.max():
            warm_inputs = prepare_window_inputs(df_for_warm, warm_start, warm_end, H_LOOK)
            ev_warm = build_ev_schedules_for_span(
                warm_inputs["index_span"],
                deterministic_actual=USE_DETERMINISTIC_EV_SCENARIO,
                rng_seed=RNG_SEED
            )
            warm_seed = build_route_aware_warmstart_seed(nominal_day_sessions, NUM_BUSES)
            p_diss_plan_nominal_full = build_time_varying_bus_dissipation_profile(
                warm_inputs["index_span"], ev_warm["nominal_sessions_full"], NUM_BUSES, P_DISS_AVG_MODE
            )
            p_diss_actual_full = build_time_varying_bus_dissipation_profile(
                warm_inputs["index_span"], ev_warm["actual_sessions_full"], NUM_BUSES, P_DISS_AVG_MODE
            )

            # Paper-aligned three-month GP for the warm window (falls back if <3 months exist).
            gp_model_warm = fit_gp_three_month(
                df_for_warm, warm_inputs["index_window"][0], seed=RNG_SEED, verbose=False
            )

            warm_results = run_smpc_month_alpha(
                T=len(warm_inputs["index_window"]), H=H_LOOK, dt=1.0,
                df_full=df_for_warm, index_span=warm_inputs["index_span"],
                P_pv_max_full=warm_inputs["P_pv_max_full"],
                P_grid_cap_full=warm_inputs["P_grid_cap_full"],
                c_elec_full=warm_inputs["c_elec_full"], c_gas_full=warm_inputs["c_gas_full"],
                E_B_min=E_B_min, E_B_max=E_B_max, P_B_max=P_B_max, eta_ch=eta_ch, eta_dc=eta_dc,
                eta_g=eta_g, rho_deg=rho_deg, rho_curt=rho_curt, rho_shed=rho_shed,
                z_nom_full=ev_warm["z_nom_full"], pmax_nom_full=ev_warm["pmax_nom_full"],
                z_actual_full=ev_warm["z_actual_full"], pmax_actual_full=ev_warm["pmax_actual_full"],
                dep_req_actual_full=ev_warm["dep_req_actual_full"],
                dep_event_count_full=ev_warm["dep_event_count_full"],
                p_diss_plan_nominal_full=p_diss_plan_nominal_full,
                p_diss_actual_full=p_diss_actual_full,
                alpha_buffer=alpha_buffer,
                E_B0=800.0,
                E_bus0_init=warm_seed,
                gp_model=gp_model_warm,
                seed_outage=RNG_SEED,
                M_scen=M_SCEN,
                noise_scale=NOISE_SCALE,
                verbose=False,
            )
            init_method = "warm_start_dec2023_same_smpc_model" if warm_tag == "dec_2023_wrap" else "warm_start_same_smpc_model"
            return np.asarray(warm_results["E_bus_real"][-1, :], float), init_method, 0.0, warm_seed

    E_bus0_init, init_lp_slack = solve_periodic_initial_bus_energy(
        day_sessions=nominal_day_sessions,
        p_diss_avg=p_diss_avg,
        alpha_buffer=alpha_buffer,
        dt=1.0,
        n_buses=NUM_BUSES,
        e_min=EV_MIN_KWH,
        e_max=EV_BATT_KWH,
    )
    return E_bus0_init, "periodic_lp_fallback_only", init_lp_slack, None


def solve_periodic_initial_bus_energy(day_sessions: pd.DataFrame,
                                      p_diss_avg: np.ndarray,
                                      alpha_buffer: float,
                                      dt: float = 1.0,
                                      n_buses: int = NUM_BUSES,
                                      e_min: float = EV_MIN_KWH,
                                      e_max: float = EV_BATT_KWH):
    """
    One-day periodic EV feasibility LP under the SAME per-bus model used in the SMPC,
    used only to seed the initial bus state when measured E0 is unavailable. Its
    internal weights affect only the seed, not any reported cost.
    """
    z_day, pmax_day = build_bus_availability_arrays(pd.date_range("2023-01-01", periods=24, freq="H"), day_sessions, n_buses)
    N = z_day.shape[1]
    m = gp.Model("periodic_initial_bus_energy")
    m.Params.OutputFlag = 0

    E = m.addVars(n_buses, N + 1, lb=0.0, ub=e_max, name="E")
    p = m.addVars(n_buses, N, lb=0.0, name="p")
    xi = m.addVars(n_buses, N + 1, lb=0.0, ub=e_min, name="xi")
    floor_clip = m.addVars(n_buses, N, lb=0.0, name="floor_clip")

    for b in range(n_buses):
        m.addConstr(E[b, 0] == E[b, N], name=f"periodic_{b}")
        for k in range(N):
            m.addConstr(p[b, k] <= float(pmax_day[b, k]) * float(z_day[b, k]))
            loss = (1.0 + float(alpha_buffer)) * dt * float(p_diss_avg[b]) * (1.0 - float(z_day[b, k]))
            m.addConstr(floor_clip[b, k] <= loss + 1e-9)
            m.addConstr(E[b, k + 1] == E[b, k] + dt * EV_CHG_EFF * p[b, k] - loss + floor_clip[b, k])
            m.addConstr(E[b, k] >= e_min - xi[b, k])
            m.addConstr(E[b, k] <= e_max)
        m.addConstr(E[b, N] >= e_min - xi[b, N])
        m.addConstr(E[b, N] <= e_max)

    obj = 1000.0 * gp.quicksum(floor_clip[b, k] for b in range(n_buses) for k in range(N)) \
        + gp.quicksum(xi[b, k] for b in range(n_buses) for k in range(N + 1)) \
        + 1e-4 * gp.quicksum(E[b, k] for b in range(n_buses) for k in range(N + 1))
    m.setObjective(obj, GRB.MINIMIZE)
    m.optimize()

    if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
        raise RuntimeError(f"Periodic initial-energy LP failed with status {m.Status}")

    E0 = np.array([E[b, 0].X for b in range(n_buses)], dtype=float)
    xi_sum = float(sum(xi[b, k].X for b in range(n_buses) for k in range(N + 1)))
    return np.clip(E0, 0.0, e_max), xi_sum


# =========================================================
# GP scenario model (PAPER-ALIGNED, Section IV-A)
#
# Three-month sliding window: the GP is CONDITIONED ("trained") on the second
# and third preceding months (M-2, M-3); kernel hyperparameters are SELECTED by
# maximizing the held-out log predictive likelihood on the most recent preceding
# month (M-1), which is excluded from conditioning. Load-trajectory scenarios are
# drawn JOINTLY from the GP posterior (sample_y), so they carry the temporal
# correlation encoded by the kernel rather than independent per-hour noise.
# =========================================================
_HOLIDAY_CACHE = {}


def _holiday_set_for_years(years):
    key = tuple(sorted({int(y) for y in years}))
    if key in _HOLIDAY_CACHE:
        return _HOLIDAY_CACHE[key]
    s = set()
    if USE_HOLIDAYS:
        try:
            import holidays as _hol
            s = set(_hol.country_holidays(HOLIDAY_COUNTRY, years=list(key)).keys())
        except Exception:
            s = set()  # package missing -> weekend-only fallback
    _HOLIDAY_CACHE[key] = s
    return s


def _time_features(index_like: pd.DatetimeIndex):
    """
    Feature map phi: cyclic hour-of-day and day-of-year encodings plus a boolean
    flag that is 1 on weekends OR public holidays. The holiday component requires
    the optional 'holidays' package; without it the flag reduces to weekends only.
    """
    idx = pd.DatetimeIndex(index_like)
    hour = idx.hour.values.astype(float)
    doy = idx.dayofyear.values.astype(float)
    sinH = np.sin(2 * np.pi * hour / 24.0)
    cosH = np.cos(2 * np.pi * hour / 24.0)
    sinY = np.sin(2 * np.pi * doy / 365.0)
    cosY = np.cos(2 * np.pi * doy / 365.0)
    is_wkend = np.asarray(idx.weekday >= 5)
    hol = _holiday_set_for_years(idx.year.unique().tolist())
    if hol:
        is_hol = np.array([d.date() in hol for d in idx], dtype=bool)
    else:
        is_hol = np.zeros(len(idx), dtype=bool)
    flag = (is_wkend | is_hol).astype(float)
    return np.c_[sinH, cosH, sinY, cosY, flag]


def _gp_kernel_template():
    """Composite Exp-Sine-Squared (periodicity) + White-noise kernel."""
    return (C(1.0, (1e-3, 1e3)) *
            ExpSineSquared(length_scale=1.0, periodicity=24.0,
                           length_scale_bounds=(1e-2, 1e3),
                           periodicity_bounds=(12.0, 1000.0))
            + WhiteKernel(noise_level=0.5, noise_level_bounds=(1e-6, 1e1)))


class FittedGP:
    """
    Thin wrapper around a fitted GaussianProcessRegressor that applies the feature
    StandardScaler before predict/sample_y. predict/sample_y return values in
    original load units (kW). sample_y draws JOINT posterior trajectories.
    """
    def __init__(self, x_scaler, gpr, meta=None):
        self.x_scaler = x_scaler
        self.gpr = gpr
        self.meta = meta or {}

    def predict(self, X, return_std=False, return_cov=False):
        Xs = self.x_scaler.transform(X)
        return self.gpr.predict(Xs, return_std=return_std, return_cov=return_cov)

    def sample_y(self, X, n_samples=1, random_state=None):
        Xs = self.x_scaler.transform(X)
        return self.gpr.sample_y(Xs, n_samples=n_samples, random_state=random_state)


def _subsample_xy(X, y, cap, seed):
    if cap is None or len(y) <= cap:
        return X, y
    r = np.random.default_rng(seed)
    sel = np.sort(r.choice(len(y), int(cap), replace=False))
    return X[sel], y[sel]


def fit_gp_fallback(history_df: pd.DataFrame, max_train=1000, recent_days=45, seed=RNG_SEED):
    """
    Fallback used ONLY when fewer than three preceding months are available:
    fit on the most recent `recent_days` of prior data with standard LML
    hyperparameter optimization. Wrapped in FittedGP so joint sampling still works.
    """
    hist = history_df.dropna(subset=["Load_kW"]).copy()
    if len(hist) == 0:
        return None
    if recent_days is not None:
        cutoff = hist.index.max() - pd.Timedelta(days=recent_days)
        hist = hist.loc[hist.index >= cutoff]
    if len(hist) > max_train:
        hist = hist.sample(max_train, random_state=seed).sort_index()
    y = hist["Load_kW"].astype(float).values
    X = _time_features(hist.index)
    xs = StandardScaler().fit(X)
    gpr = GaussianProcessRegressor(
        kernel=_gp_kernel_template(), alpha=_GP_JITTER, normalize_y=True,
        n_restarts_optimizer=0, random_state=seed
    )
    gpr.fit(xs.transform(X), y)
    return FittedGP(xs, gpr, meta={
        "mode": "fallback",
        "train_span": (str(hist.index.min()), str(hist.index.max())),
        "val_span": None, "n_train": int(len(hist)), "kernel": str(gpr.kernel_),
    })


def fit_gp_three_month(df: pd.DataFrame, target_month_start,
                       opt_train_cap=GP_OPT_TRAIN_CAP, opt_val_cap=GP_OPT_VAL_CAP,
                       final_train_cap=GP_FINAL_TRAIN_CAP, n_restarts=GP_N_RESTARTS,
                       seed=RNG_SEED, verbose=False):
    """
    Paper-aligned three-month sliding-window GP fit (Section IV-A).

    - Conditioning ("training") data: the SECOND and THIRD preceding months
      (M-2, M-3) relative to `target_month_start` (= forecast month M).
    - Hyperparameter selection: kernel hyperparameters theta are chosen to
      MAXIMIZE the held-out Gaussian log-likelihood on the MOST RECENT preceding
      month (M-1), which is excluded from conditioning. (scikit-learn has no
      native train/validation split, so this is an explicit search: for each
      theta, condition a GP on M-2,M-3 and score the log-likelihood of M-1.)
    - The returned model conditions on M-2,M-3 with the selected theta and
      predicts / samples in original kW units.

    Falls back to `fit_gp_fallback` if three preceding months are unavailable.
    """
    Mstart = pd.Timestamp(target_month_start).normalize().replace(day=1)
    M1start = Mstart - pd.DateOffset(months=1)   # most recent preceding month -> validation
    M3start = Mstart - pd.DateOffset(months=3)   # start of the conditioning window
    idx = df.index

    train = df.loc[(idx >= M3start) & (idx < M1start)].dropna(subset=["Load_kW"])  # M-2 and M-3
    val = df.loc[(idx >= M1start) & (idx < Mstart)].dropna(subset=["Load_kW"])      # M-1

    if len(train) < 50 or len(val) < 50:
        return fit_gp_fallback(df.loc[idx < Mstart], seed=seed)

    Xtr_all = _time_features(train.index)
    ytr_all = train["Load_kW"].to_numpy(float)
    Xva_all = _time_features(val.index)
    yva_all = val["Load_kW"].to_numpy(float)

    xs = StandardScaler().fit(Xtr_all)
    Xo, yo = _subsample_xy(xs.transform(Xtr_all), ytr_all, opt_train_cap, seed)
    Xv, yv = _subsample_xy(xs.transform(Xva_all), yva_all, opt_val_cap, seed + 1)

    ymu, ystd = float(yo.mean()), float(yo.std() + 1e-9)
    yo_n = (yo - ymu) / ystd
    yv_n = (yv - ymu) / ystd

    tmpl = _gp_kernel_template()
    bounds = tmpl.bounds  # log-space hyperparameter bounds

    def neg_val_ll(theta):
        try:
            k = tmpl.clone_with_theta(theta)
            gpr = GaussianProcessRegressor(kernel=k, alpha=_GP_JITTER, optimizer=None,
                                           normalize_y=False, copy_X_train=False)
            gpr.fit(Xo, yo_n)
            mu, sd = gpr.predict(Xv, return_std=True)
        except Exception:
            return 1e25
        sd = np.maximum(sd, 1e-6)
        nll = 0.5 * np.sum(((yv_n - mu) / sd) ** 2 + np.log(2.0 * np.pi * sd ** 2))
        return float(nll) if np.isfinite(nll) else 1e25

    r = np.random.default_rng(seed + 7)
    starts = [tmpl.theta] + [np.array([r.uniform(b[0], b[1]) for b in bounds])
                             for _ in range(int(max(0, n_restarts)))]
    best_theta, best_f = None, np.inf
    for s0 in starts:
        try:
            res = minimize(neg_val_ll, s0, method="L-BFGS-B", bounds=bounds)
            if res.fun < best_f:
                best_f, best_theta = float(res.fun), res.x
        except Exception:
            continue
    if best_theta is None:
        best_theta = tmpl.theta

    sel_kernel = tmpl.clone_with_theta(best_theta)
    Xf, yf = _subsample_xy(Xtr_all, ytr_all, final_train_cap, seed + 3)
    final = GaussianProcessRegressor(kernel=sel_kernel, alpha=_GP_JITTER,
                                     optimizer=None, normalize_y=True)
    final.fit(xs.transform(Xf), yf)

    meta = {
        "mode": "three_month",
        "train_span": (str(train.index.min()), str(train.index.max())),
        "val_span": (str(val.index.min()), str(val.index.max())),
        "n_train": int(len(train)), "n_val": int(len(val)),
        "val_nll": float(best_f), "kernel": str(final.kernel_),
    }
    return FittedGP(xs, final, meta)


def generate_load_scenarios_gp_with_model(index_window, gp_model, M, current_load,
                                          noise_scale=1.0, seed=RNG_SEED):
    """
    Draw M load-trajectory scenarios over the horizon. With a fitted GP the
    scenarios are JOINT posterior samples (sample_y), preserving the temporal
    correlation encoded by the kernel. The first step is anchored to the measured
    current load, and all values are clipped to be non-negative. `noise_scale` is
    used only by the GP-less fallback path.
    """
    X_win = _time_features(pd.DatetimeIndex(index_window))
    H = len(X_win)
    if gp_model is None:
        rng = np.random.default_rng(seed)
        mu = np.full(H, current_load, dtype=float)
        sd = np.full(H, max(1e-3, 0.05 * max(1.0, current_load)), dtype=float)
        L = rng.normal(mu, noise_scale * sd, size=(M, H))
    else:
        mu, sd = gp_model.predict(X_win, return_std=True)
        sd = np.maximum(sd, 1e-3)
        L = np.asarray(gp_model.sample_y(X_win, n_samples=M, random_state=seed)).T  # (M, H) joint draws
    L = np.maximum(L, 0.0)
    L[:, 0] = current_load
    return L, mu, sd


# =========================================================
# Outage scenarios (Eq. 10)
# =========================================================
def markov_outage_scenarios_window(h, M, psi_on_off_per_h, nu_off_on_per_h, current_state, dt=1.0, seed=1234):
    """
    Two-state continuous-time Markov chain for grid availability (Eq. 10).
    State 1 = grid available (ON), 0 = outage (OFF).
        p_{ON->OFF} = 1 - exp(-psi * dt)   (psi  = outage rate)
        p_{OFF->ON} = 1 - exp(-nu  * dt)   (nu   = restoration rate)
    Steady-state availability = nu / (psi + nu); mean outage duration = 1 / nu.
    With psi=0.10/h, nu=0.50/h: availability = 0.5/0.6 = 0.833, mean outage = 2 h.
    """
    rng = np.random.default_rng(seed)
    p_on_off = 1.0 - np.exp(-float(psi_on_off_per_h) * float(dt))
    p_off_on = 1.0 - np.exp(-float(nu_off_on_per_h) * float(dt))
    Theta = np.zeros((M, h), dtype=int)
    Theta[:, 0] = int(current_state)
    for s in range(M):
        for k in range(1, h):
            prev = Theta[s, k - 1]
            if prev == 1:  # ON -> OFF
                Theta[s, k] = 0 if (rng.random() < p_on_off) else 1
            else:          # OFF -> ON
                Theta[s, k] = 1 if (rng.random() < p_off_on) else 0
    return Theta


# =========================================================
# SMPC SOLVER: per-bus EV buffer model (paper-aligned objective)
# =========================================================
def solve_energy_hub_smpc_ev_buffer(
    N, M, dt,
    L_scenarios,
    P_pv_max, c_elec, c_gas,
    P_grid_cap,
    E_B_min, E_B_max, P_B_max, eta_ch, eta_dc,
    E_bus0_vec,
    z_plan, pmax_plan, p_diss_plan,
    alpha_buffer,
    grid_avail_scenarios=None,
    grid_cap_now=None,
    eta_g=0.35,
    rho_deg=0.02, rho_curt=0.05, rho_shed=1000.0,
    rho_ev_slack=RHO_EV_SLACK,
    rho_ev_floor_clip=RHO_EV_FLOOR_CLIP,
    lambda_ev_ramp=LAMBDA_EV_RAMP_Q,
    lambda_ev_peak=LAMBDA_EV_PEAK_Q,
    E_B0=800.0,
    E_B_terminal_target=None,
    rho_terminal=1e-3,
    verbose=False,
):
    assert L_scenarios.shape == (M, N)
    B = z_plan.shape[0]

    P_pv_max = np.asarray(P_pv_max, float)
    c_elec = np.asarray(c_elec, float)
    c_gas = np.asarray(c_gas, float)
    P_grid_cap = np.asarray(P_grid_cap, float)
    z_plan = np.asarray(z_plan, float)
    pmax_plan = np.asarray(pmax_plan, float)
    p_diss_plan = np.asarray(p_diss_plan, float)
    if p_diss_plan.ndim == 1:
        p_diss_plan = np.repeat(p_diss_plan.reshape(-1, 1), N, axis=1)
    elif p_diss_plan.ndim == 2 and p_diss_plan.shape == (B, N):
        p_diss_plan = p_diss_plan.astype(float)
    else:
        raise ValueError(f"p_diss_plan must have shape ({B},) or ({B}, {N}), got {p_diss_plan.shape}")
    E_bus0_vec = np.asarray(E_bus0_vec, float).reshape(-1)

    if grid_avail_scenarios is None:
        grid_avail_scenarios = np.ones((M, N), dtype=int)
    else:
        grid_avail_scenarios = np.asarray(grid_avail_scenarios, int)

    m = gp.Model("SMPC_EV_PerBus_Buffer")
    m.Params.OutputFlag = 0
    m.Params.Presolve = 2
    m.Params.DualReductions = 0

    shape = (M, N)
    E_B = m.addMVar(shape=(M, N + 1), lb=E_B_min, ub=E_B_max, name="E_B")
    P_B_ch = m.addMVar(shape=shape, lb=0.0, ub=P_B_max, name="P_B_ch")
    P_B_dc_ub = P_B_max if ALLOW_BESS_DISCHARGE else 0.0
    P_B_dc = m.addMVar(shape=shape, lb=0.0, ub=P_B_dc_ub, name="P_B_dc")
    P_pv = m.addMVar(shape=shape, lb=0.0, name="P_pv")
    P_g = m.addMVar(shape=shape, lb=0.0, name="P_g")
    P_e = m.addMVar(shape=shape, lb=0.0, name="P_e")
    P_EV_ch = m.addMVar(shape=shape, lb=0.0, name="P_EV_ch")
    LS = m.addMVar(shape=shape, lb=0.0, name="LoadShed")

    P_e0 = m.addVar(lb=0.0, name="P_e0")
    P_g0 = m.addVar(lb=0.0, name="P_g0")
    P_pv0 = m.addVar(lb=0.0, name="P_pv0")
    P_B_ch0 = m.addVar(lb=0.0, ub=P_B_max, name="P_B_ch0")
    P_B_dc0 = m.addVar(lb=0.0, ub=P_B_dc_ub, name="P_B_dc0")
    P_EV_ch0 = m.addVar(lb=0.0, name="P_EV_ch0")
    LS0 = m.addVar(lb=0.0, name="LS0")

    p_bus = m.addVars(M, B, N, lb=0.0, name="p_bus")
    p_bus0 = m.addVars(B, lb=0.0, name="p_bus0")
    E_bus = m.addVars(M, B, N + 1, lb=0.0, ub=EV_BATT_KWH, name="E_bus")
    xi_bus = m.addVars(M, B, N + 1, lb=0.0, ub=EV_MIN_KWH, name="xi_bus")
    floor_clip = m.addVars(M, B, N, lb=0.0, name="floor_clip")

    for s in range(M):
        m.addConstr(P_e[s, 0] == P_e0)
        m.addConstr(P_g[s, 0] == P_g0)
        m.addConstr(P_pv[s, 0] == P_pv0)
        m.addConstr(P_B_ch[s, 0] == P_B_ch0)
        m.addConstr(P_B_dc[s, 0] == P_B_dc0)
        m.addConstr(P_EV_ch[s, 0] == P_EV_ch0)
        m.addConstr(LS[s, 0] == LS0)
        m.addConstr(E_B[s, 0] == E_B0)

    for b in range(B):
        m.addConstr(p_bus0[b] <= float(pmax_plan[b, 0]) * float(z_plan[b, 0]))
    m.addConstr(gp.quicksum(p_bus0[b] for b in range(B)) == P_EV_ch0)

    grid_cap_now = float(P_grid_cap[0]) if grid_cap_now is None else float(grid_cap_now)
    m.addConstr(P_e0 <= grid_cap_now, name="GridCapReality_k0")

    for s in range(M):
        for b in range(B):
            m.addConstr(E_bus[s, b, 0] == float(E_bus0_vec[b]))
            m.addConstr(p_bus[s, b, 0] == p_bus0[b])
            m.addConstr(E_bus[s, b, 0] >= EV_MIN_KWH - xi_bus[s, b, 0])
            for k in range(N):
                m.addConstr(p_bus[s, b, k] <= float(pmax_plan[b, k]) * float(z_plan[b, k]))
                loss = (1.0 + float(alpha_buffer)) * dt * float(p_diss_plan[b, k]) * (1.0 - float(z_plan[b, k]))
                # floor_clip keeps the optimizer feasible when realized depletion would
                # drive the bus below 0 kWh before it can recharge; its activation is
                # heavily penalized and reported.
                m.addConstr(floor_clip[s, b, k] <= loss + 1e-9)
                m.addConstr(
                    E_bus[s, b, k + 1] == E_bus[s, b, k] + dt * EV_CHG_EFF * p_bus[s, b, k] - loss + floor_clip[s, b, k]
                )
                m.addConstr(E_bus[s, b, k] >= EV_MIN_KWH - xi_bus[s, b, k])
                m.addConstr(E_bus[s, b, k] <= EV_BATT_KWH)
            m.addConstr(E_bus[s, b, N] >= EV_MIN_KWH - xi_bus[s, b, N])
            m.addConstr(E_bus[s, b, N] <= EV_BATT_KWH)

        for k in range(N):
            m.addConstr(P_EV_ch[s, k] == gp.quicksum(p_bus[s, b, k] for b in range(B)))

            m.addConstr(
                E_B[s, k + 1] == E_B[s, k] + dt * (eta_ch * P_B_ch[s, k] - (1.0 / eta_dc) * P_B_dc[s, k])
            )
            m.addConstr(P_pv[s, k] <= P_pv_max[k])
            if k >= 1:
                m.addConstr(P_e[s, k] <= grid_avail_scenarios[s, k] * float(P_grid_cap[k]))

            m.addConstr(
                L_scenarios[s, k]
                == P_e[s, k] + P_g[s, k] + P_pv[s, k] + P_B_dc[s, k]
                 - P_B_ch[s, k] - P_EV_ch[s, k] + LS[s, k]
            )

    # ---- Terminal cost (Eq. 14): one-sided rho_term * max(E_ref - E_N, 0)^2 ----
    term_cost = 0.0
    if E_B_terminal_target is not None:
        term_short = m.addMVar(shape=(M,), lb=0.0, name="term_short")
        m.addConstr(term_short >= float(E_B_terminal_target) - E_B[:, N])
        term_cost = rho_terminal * (term_short @ term_short) / float(M)

    invM = 1.0 / float(M)
    grid_cost = dt * gp.quicksum(c_elec[k] * gp.quicksum(P_e[s, k] for s in range(M)) for k in range(N)) * invM
    # Diesel cost: c_gas is already per-electric-kWh (Table I c^d), so no extra /eta_g.
    gas_cost = dt * gp.quicksum(c_gas[k] * gp.quicksum(P_g[s, k] for s in range(M)) for k in range(N)) * invM
    # Battery degradation (Eq. 13): discharge only.
    degr_cost = dt * rho_deg * gp.quicksum(gp.quicksum(P_B_dc[s, k] for s in range(M)) for k in range(N)) * invM
    curt_cost = dt * rho_curt * gp.quicksum(gp.quicksum(P_pv_max[k] - P_pv[s, k] for s in range(M)) for k in range(N)) * invM
    shed_cost = dt * rho_shed * gp.quicksum(gp.quicksum(LS[s, k] for s in range(M)) for k in range(N)) * invM
    ev_slack_cost = rho_ev_slack * gp.quicksum(xi_bus[s, b, k] for s in range(M) for b in range(B) for k in range(N + 1)) * invM
    ev_floor_cost = rho_ev_floor_clip * gp.quicksum(floor_clip[s, b, k] for s in range(M) for b in range(B) for k in range(N)) * invM

    # ---- EV regularization (Eq. 16): per-bus QUADRATIC ramp + peak ----
    # Ramp at k=0 is zero (p_bus[s,b,0] == p_bus0[b]); penalize k = 1..N-1.
    ev_ramp_cost = lambda_ev_ramp * gp.quicksum(
        (p_bus[s, b, k] - p_bus[s, b, k - 1]) * (p_bus[s, b, k] - p_bus[s, b, k - 1])
        for s in range(M) for b in range(B) for k in range(1, N)
    ) * invM
    ev_peak_cost = lambda_ev_peak * gp.quicksum(
        p_bus[s, b, k] * p_bus[s, b, k]
        for s in range(M) for b in range(B) for k in range(N)
    ) * invM

    m.setObjective(
        grid_cost + gas_cost + degr_cost + curt_cost + shed_cost + term_cost
        + ev_slack_cost + ev_floor_cost + ev_ramp_cost + ev_peak_cost,
        GRB.MINIMIZE
    )
    m.optimize()

    if m.Status == GRB.INF_OR_UNBD:
        m.reset()
        m.Params.DualReductions = 0
        m.optimize()
    if m.Status == GRB.INFEASIBLE:
        try:
            m.computeIIS()
            m.write("smpc_ev_buffer_infeasible.ilp")
        except Exception:
            pass
    if m.Status not in (GRB.OPTIMAL, GRB.SUBOPTIMAL):
        raise RuntimeError(f"Gurobi status: {m.Status}")

    p_bus0_opt = np.array([p_bus0[b].X for b in range(B)], dtype=float)
    E_bus_next_plan = np.array([E_bus[0, b, 1].X for b in range(B)], dtype=float)
    xi_now_bus = np.array([xi_bus[0, b, 0].X for b in range(B)], dtype=float)
    xi_next_bus = np.array([xi_bus[0, b, 1].X for b in range(B)], dtype=float)
    floor_clip_now_bus = np.array([floor_clip[0, b, 0].X for b in range(B)], dtype=float)

    actions0 = dict(
        P_e=float(P_e0.X),
        P_g=float(P_g0.X),
        P_pv=float(P_pv0.X),
        P_B_ch=float(P_B_ch0.X),
        P_B_dc=float(P_B_dc0.X),
        P_EV_ch=float(P_EV_ch0.X),
        LS=float(LS0.X),
        GRID_CAP_NOW=float(grid_cap_now),
        p_bus0_opt_kW=p_bus0_opt,
        E_bus_next_plan_kWh=E_bus_next_plan,
        xi_now_bus_kWh=xi_now_bus,
        xi_next_bus_kWh=xi_next_bus,
        xi_next_total_kWh=float(np.sum(xi_next_bus)),
        floor_clip_now_bus_kWh=floor_clip_now_bus,
        floor_clip_now_total_kWh=float(np.sum(floor_clip_now_bus)),
    )

    return dict(status=m.Status, objective=float(m.ObjVal), actions0=actions0,
                solve_time_s=float(m.Runtime))


# =========================================================
# Receding-horizon driver
# =========================================================

def run_smpc_month_alpha(
    T, H, dt,
    df_full, index_span,
    P_pv_max_full, P_grid_cap_full, c_elec_full, c_gas_full,
    E_B_min, E_B_max, P_B_max, eta_ch, eta_dc,
    eta_g, rho_deg, rho_curt, rho_shed,
    z_nom_full, pmax_nom_full,
    z_actual_full, pmax_actual_full,
    dep_req_actual_full, dep_event_count_full,
    p_diss_plan_nominal_full,
    p_diss_actual_full,
    alpha_buffer,
    E_B0,
    E_bus0_init,
    gp_model,
    seed_outage=RNG_SEED,
    M_scen=M_SCEN,
    noise_scale=NOISE_SCALE,
    verbose=False,
):
    P_load_real_full = df_full.loc[index_span, "Load_kW"].to_numpy(float)
    grid_up_reality_full = (P_grid_cap_full > 0.0).astype(int)
    B = z_nom_full.shape[0]

    p_diss_plan_nominal_full = np.asarray(p_diss_plan_nominal_full, float)
    p_diss_actual_full = np.asarray(p_diss_actual_full, float)
    if p_diss_plan_nominal_full.shape[0] != B or p_diss_actual_full.shape[0] != B:
        raise ValueError("Dissipation profiles must have NUM_BUSES rows.")

    P_e_app = np.zeros(T)
    P_g_app = np.zeros(T)
    P_pv_app = np.zeros(T)
    P_B_ch_app = np.zeros(T)
    P_B_dc_app = np.zeros(T)
    P_EV_ch_app = np.zeros(T)
    LS_app = np.zeros(T)

    E_B_real = np.zeros(T + 1)
    E_B_real[0] = E_B0
    E_bus_real = np.zeros((T + 1, B), dtype=float)
    E_bus_real[0, :] = np.asarray(E_bus0_init, float)

    E_bus_plan_nodes = np.zeros((T + 1, B), dtype=float)
    E_bus_plan_nodes[0, :] = np.asarray(E_bus0_init, float)
    xi_plan_nodes = np.zeros((T + 1, B), dtype=float)
    xi_plan_nodes[0, :] = np.maximum(EV_MIN_KWH - E_bus0_init, 0.0)

    grid_cost_k = np.zeros(T)
    gas_cost_k = np.zeros(T)
    degr_cost_k = np.zeros(T)
    curt_cost_k = np.zeros(T)
    shed_cost_k = np.zeros(T)
    ev_planned_slack_cost_k = np.zeros(T)
    ev_floor_clip_cost_k = np.zeros(T)
    ev_floor_clip_kWh_k = np.zeros(T)
    ev_actual_short_kWh_k1 = np.zeros(T)
    ev_planned_short_kWh_k1 = np.zeros(T)
    ev_departure_short_k = np.zeros(T)
    ev_departure_viol_events_k = np.zeros(T, dtype=int)
    ev_departure_active_events_k = np.zeros(T, dtype=int)
    solve_time_k = np.zeros(T)        # Gurobi QP solve time per step [s]
    step_wall_k = np.zeros(T)         # wall-clock build+solve+extract per step [s]

    rng = np.random.default_rng(seed_outage)

    for t in range(T):
        dep_req_now = np.asarray(dep_req_actual_full[:, t], float)
        dep_active_now = np.asarray(dep_event_count_full[:, t] > 0, bool)
        ev_departure_short_by_bus = np.maximum(dep_req_now - E_bus_real[t, :], 0.0) * dep_active_now.astype(float)
        ev_departure_short_k[t] = float(np.sum(ev_departure_short_by_bus))
        ev_departure_viol_events_k[t] = int(np.sum((ev_departure_short_by_bus > 1e-6) & dep_active_now))
        ev_departure_active_events_k[t] = int(np.sum(dep_active_now))

        win_index = pd.DatetimeIndex(index_span)[t:t + H]
        current_load = float(P_load_real_full[t])
        grid_cap_now = float(P_grid_cap_full[t])
        grid_state_now = int(grid_up_reality_full[t])

        L_scen, _, _ = generate_load_scenarios_gp_with_model(
            win_index, gp_model, M_scen, current_load, noise_scale=noise_scale, seed=RNG_SEED + t
        )
        base_seed = int(rng.integers(0, 2**31 - 1))
        Theta_scen = markov_outage_scenarios_window(
            H, M_scen, OUT_LAMBDA, OUT_MU, current_state=grid_state_now, dt=dt, seed=base_seed
        )

        if PLAN_WITH_ACTUAL_EV_FULL_HORIZON:
            z_plan = np.asarray(z_actual_full[:, t:t + H], float).copy()
            pmax_plan = np.asarray(pmax_actual_full[:, t:t + H], float).copy()
        else:
            z_plan = np.asarray(z_nom_full[:, t:t + H], float).copy()
            pmax_plan = np.asarray(pmax_nom_full[:, t:t + H], float).copy()
            z_plan[:, 0] = z_actual_full[:, t]
            pmax_plan[:, 0] = pmax_actual_full[:, t]

        p_diss_plan_win = np.asarray(p_diss_plan_nominal_full[:, t:t + H], float)

        _t_wall0 = time.perf_counter()
        sol = solve_energy_hub_smpc_ev_buffer(
            N=H, M=M_scen, dt=dt,
            L_scenarios=L_scen,
            P_pv_max=P_pv_max_full[t:t + H],
            c_elec=c_elec_full[t:t + H],
            c_gas=c_gas_full[t:t + H],
            P_grid_cap=P_grid_cap_full[t:t + H],
            E_B_min=E_B_min, E_B_max=E_B_max, P_B_max=P_B_max, eta_ch=eta_ch, eta_dc=eta_dc,
            E_bus0_vec=E_bus_real[t, :],
            z_plan=z_plan, pmax_plan=pmax_plan, p_diss_plan=p_diss_plan_win,
            alpha_buffer=alpha_buffer,
            grid_avail_scenarios=Theta_scen,
            grid_cap_now=grid_cap_now,
            eta_g=eta_g,
            rho_deg=rho_deg, rho_curt=rho_curt, rho_shed=rho_shed,
            rho_ev_slack=RHO_EV_SLACK,
            rho_ev_floor_clip=RHO_EV_FLOOR_CLIP,
            lambda_ev_ramp=LAMBDA_EV_RAMP_Q,
            lambda_ev_peak=LAMBDA_EV_PEAK_Q,
            E_B0=E_B_real[t],
            E_B_terminal_target=E_B_real[0],
            rho_terminal=1e-3,
            verbose=verbose,
        )
        a0 = sol["actions0"]
        step_wall_k[t] = time.perf_counter() - _t_wall0
        solve_time_k[t] = float(sol.get("solve_time_s", np.nan))

        P_e_app[t] = a0["P_e"]
        P_g_app[t] = a0["P_g"]
        P_pv_app[t] = a0["P_pv"]
        P_B_ch_app[t] = a0["P_B_ch"]
        P_B_dc_app[t] = a0["P_B_dc"]
        LS_app[t] = a0["LS"]

        p_bus_apply = np.asarray(a0["p_bus0_opt_kW"], float).copy()
        p_bus_apply = np.maximum(p_bus_apply, 0.0)
        p_bus_apply = np.minimum(p_bus_apply, pmax_actual_full[:, t] * z_actual_full[:, t])
        p_cap_energy = np.maximum(EV_BATT_KWH - E_bus_real[t, :], 0.0) / max(dt * EV_CHG_EFF, 1e-9)
        p_bus_apply = np.minimum(p_bus_apply, p_cap_energy)
        P_EV_ch_app[t] = float(np.sum(p_bus_apply))

        loss_actual = dt * p_diss_actual_full[:, t] * (1.0 - z_actual_full[:, t])
        E_bus_real[t + 1, :] = np.clip(E_bus_real[t, :] + dt * EV_CHG_EFF * p_bus_apply - loss_actual, 0.0, EV_BATT_KWH)

        E_bus_plan_nodes[t + 1, :] = np.asarray(a0["E_bus_next_plan_kWh"], float)
        xi_plan_nodes[t + 1, :] = np.asarray(a0["xi_next_bus_kWh"], float)

        E_B_real[t + 1] = E_B_real[t] + dt * (eta_ch * P_B_ch_app[t] - (1.0 / eta_dc) * P_B_dc_app[t])

        grid_cost_k[t] = c_elec_full[t] * P_e_app[t] * dt
        gas_cost_k[t] = c_gas_full[t] * P_g_app[t] * dt
        # Battery degradation accounting (Eq. 13): discharge only.
        degr_cost_k[t] = rho_deg * P_B_dc_app[t] * dt
        curt_cost_k[t] = rho_curt * (P_pv_max_full[t] - P_pv_app[t]) * dt
        shed_cost_k[t] = rho_shed * LS_app[t] * dt
        ev_planned_short_kWh_k1[t] = float(np.sum(a0["xi_next_bus_kWh"]))
        ev_planned_slack_cost_k[t] = RHO_EV_SLACK * ev_planned_short_kWh_k1[t]
        ev_floor_clip_kWh_k[t] = float(a0.get("floor_clip_now_total_kWh", 0.0))
        ev_floor_clip_cost_k[t] = RHO_EV_FLOOR_CLIP * ev_floor_clip_kWh_k[t]
        ev_actual_short_kWh_k1[t] = float(np.sum(np.maximum(EV_MIN_KWH - E_bus_real[t + 1, :], 0.0)))

    return {
        "E_B": E_B_real,
        "E_bus_real": E_bus_real,
        "E_bus_plan": E_bus_plan_nodes,
        "xi_plan_nodes": xi_plan_nodes,
        "P_e": P_e_app,
        "P_g": P_g_app,
        "P_pv": P_pv_app,
        "P_B_ch": P_B_ch_app,
        "P_B_dc": P_B_dc_app,
        "P_EV_ch": P_EV_ch_app,
        "LS": LS_app,
        "grid_cost_k": grid_cost_k,
        "gas_cost_k": gas_cost_k,
        "degr_cost_k": degr_cost_k,
        "curt_cost_k": curt_cost_k,
        "shed_cost_k": shed_cost_k,
        "ev_planned_slack_cost_k": ev_planned_slack_cost_k,
        "ev_floor_clip_cost_k": ev_floor_clip_cost_k,
        "ev_floor_clip_kWh_k": ev_floor_clip_kWh_k,
        "ev_planned_short_kWh_k1": ev_planned_short_kWh_k1,
        "ev_actual_short_kWh_k1": ev_actual_short_kWh_k1,
        "ev_departure_short_k": ev_departure_short_k,
        "ev_departure_viol_events_k": ev_departure_viol_events_k,
        "ev_departure_active_events_k": ev_departure_active_events_k,
        "solve_time_k": solve_time_k,
        "step_wall_k": step_wall_k,
    }


# =========================================================
# Reporting helpers
# =========================================================

def build_reporting_frames(mon, alpha_buffer, index_month, dfe,
                           P_pv_max_full, P_grid_cap_full, c_elec_full, c_gas_full,
                           z_nom_full, pmax_nom_full, z_actual_full, pmax_actual_full,
                           dep_req_actual_full, dep_event_count_full,
                           p_diss_plan_nominal_full, p_diss_actual_full, results, dt):
    T = len(index_month)
    bus_real = np.asarray(results["E_bus_real"][:T + 1, :], float)
    bus_plan = np.asarray(results["E_bus_plan"][:T + 1, :], float)
    short_actual = np.maximum(EV_MIN_KWH - bus_real, 0.0)
    short_plan = np.maximum(EV_MIN_KWH - bus_plan, 0.0)

    dep_metrics_actual = compute_departure_violation_metrics(bus_real, dep_req_actual_full[:, :T], dep_event_count_full[:, :T])

    node_times = pd.date_range(index_month[0], periods=T + 1, freq="H")
    dep_gap_node = np.r_[dep_metrics_actual["gap_by_time_kWh"], 0.0]
    dep_viol_node = np.r_[dep_metrics_actual["viol_events_by_time"], 0]
    dep_active_node = np.r_[dep_metrics_actual["active_events_by_time"], 0]

    node_df = pd.DataFrame({
        "Datetime": node_times,
        "Month": mon,
        "Alpha_Buffer": alpha_buffer,
        "E_bus_total_actual_kWh": bus_real.sum(axis=1),
        "E_bus_total_plan_kWh": bus_plan.sum(axis=1),
        "E_bus_min_actual_kWh": bus_real.min(axis=1),
        "E_bus_min_plan_kWh": bus_plan.min(axis=1),
        "EV_actual_total_short_kWh": short_actual.sum(axis=1),
        "EV_plan_total_short_kWh": short_plan.sum(axis=1),
        "EV_actual_violating_buses": (short_actual > 1e-6).sum(axis=1),
        "EV_plan_violating_buses": (short_plan > 1e-6).sum(axis=1),
        "Departure_total_short_kWh": dep_gap_node,
        "Departure_viol_events": dep_viol_node,
        "Departure_active_events": dep_active_node,
        "E_B_kWh": np.asarray(results["E_B"][:T + 1], float),
    })

    outage_flag = (np.asarray(P_grid_cap_full[:T], float) <= 0.0).astype(int)
    diesel_ts = diesel_co2_timeseries(results["P_g"][:T], dt_h=dt)
    p_diss_nominal_total = np.asarray(p_diss_plan_nominal_full[:, :T], float).sum(axis=0)
    p_diss_actual_total = np.asarray(p_diss_actual_full[:, :T], float).sum(axis=0)

    dispatch_df = pd.DataFrame({
        "Datetime": pd.DatetimeIndex(index_month),
        "Month": mon,
        "Alpha_Buffer": alpha_buffer,
        "Load_kW": np.asarray(dfe.loc[index_month, "Load_kW"], float),
        "PV_avail_kW": np.asarray(P_pv_max_full[:T], float),
        "Grid_Cap_kW": np.asarray(P_grid_cap_full[:T], float),
        "dataset_outage_flag": outage_flag,
        "c_elec_GHS_per_kWh": np.asarray(c_elec_full[:T], float),
        "c_diesel_GHS_per_kWh_elec": np.asarray(c_gas_full[:T], float),
        "EV_nominal_plugged_buses": z_nom_full[:, :T].sum(axis=0),
        "EV_actual_plugged_buses": z_actual_full[:, :T].sum(axis=0),
        "EV_nominal_agg_cap_kW": pmax_nom_full[:, :T].sum(axis=0),
        "EV_actual_agg_cap_kW": pmax_actual_full[:, :T].sum(axis=0),
        "P_diss_total_kW": p_diss_actual_total,
        "P_diss_controller_nominal_total_kW": p_diss_nominal_total,
        "E_B_kWh_node_start": np.asarray(results["E_B"][:T], float),
        "E_B_kWh_node_end": np.asarray(results["E_B"][1:T + 1], float),
        "E_bus_total_actual_kWh_node_start": bus_real[:T, :].sum(axis=1),
        "E_bus_total_actual_kWh_node_end": bus_real[1:T + 1, :].sum(axis=1),
        "E_bus_total_plan_kWh_node_start": bus_plan[:T, :].sum(axis=1),
        "E_bus_total_plan_kWh_node_end": bus_plan[1:T + 1, :].sum(axis=1),
        "E_bus_min_actual_kWh_node_start": bus_real[:T, :].min(axis=1),
        "E_bus_min_actual_kWh_node_end": bus_real[1:T + 1, :].min(axis=1),
        "E_bus_min_plan_kWh_node_start": bus_plan[:T, :].min(axis=1),
        "E_bus_min_plan_kWh_node_end": bus_plan[1:T + 1, :].min(axis=1),
        "EV_actual_total_short_kWh_node_end": short_actual[1:T + 1, :].sum(axis=1),
        "EV_plan_total_short_kWh_node_end": short_plan[1:T + 1, :].sum(axis=1),
        "Departure_total_short_kWh_node_start": dep_metrics_actual["gap_by_time_kWh"][:T],
        "Departure_viol_events_node_start": dep_metrics_actual["viol_events_by_time"][:T],
        "Departure_active_events_node_start": dep_metrics_actual["active_events_by_time"][:T],
        "P_e_kW": np.asarray(results["P_e"][:T], float),
        "P_g_kW": np.asarray(results["P_g"][:T], float),
        "P_pv_kW": np.asarray(results["P_pv"][:T], float),
        "P_B_ch_kW": np.asarray(results["P_B_ch"][:T], float),
        "P_B_dc_kW": np.asarray(results["P_B_dc"][:T], float),
        "P_EV_ch_kW": np.asarray(results["P_EV_ch"][:T], float),
        "Load_shed_kW": np.asarray(results["LS"][:T], float),
        "grid_cost_GHS": np.asarray(results["grid_cost_k"][:T], float),
        "diesel_cost_GHS": np.asarray(results["gas_cost_k"][:T], float),
        "batt_degr_cost_GHS": np.asarray(results["degr_cost_k"][:T], float),
        "pv_curt_cost_GHS": np.asarray(results["curt_cost_k"][:T], float),
        "load_shed_cost_GHS": np.asarray(results["shed_cost_k"][:T], float),
        "ev_planned_slack_cost_GHS": np.asarray(results["ev_planned_slack_cost_k"][:T], float),
        "ev_floor_clip_cost_GHS": np.asarray(results["ev_floor_clip_cost_k"][:T], float),
        "ev_floor_clip_kWh_k": np.asarray(results["ev_floor_clip_kWh_k"][:T], float),
        "ev_planned_short_kWh_k1": np.asarray(results["ev_planned_short_kWh_k1"][:T], float),
        "ev_actual_short_kWh_k1": np.asarray(results["ev_actual_short_kWh_k1"][:T], float),
        "diesel_elec_kWh_step": diesel_ts["diesel_elec_kWh_step"],
        "diesel_co2_kg_step": diesel_ts["diesel_co2_kg_step"],
        "diesel_co2_tonnes_step": diesel_ts["diesel_co2_tonnes_step"],
        "diesel_litres_implied_step": diesel_ts["diesel_litres_implied_step"],
    })
    dispatch_df["microgrid_step_cost_GHS"] = (
        dispatch_df["grid_cost_GHS"]
        + dispatch_df["diesel_cost_GHS"]
        + dispatch_df["batt_degr_cost_GHS"]
        + dispatch_df["pv_curt_cost_GHS"]
        + dispatch_df["load_shed_cost_GHS"]
    )
    dispatch_df["total_step_cost_with_ev_GHS"] = (
        dispatch_df["microgrid_step_cost_GHS"]
        + dispatch_df["ev_planned_slack_cost_GHS"]
        + dispatch_df["ev_floor_clip_cost_GHS"]
    )
    return node_df, dispatch_df, dep_metrics_actual


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":
    H_LOOK = 24
    dt = 1.0

    for candidate in ["synthetic_ashesi_2022-09_to_2023-12_hourly_filled_pv.csv",
                      "synthetic_ashesi_2023_hourly_filled_pv.csv",
                      "synthetic_ashesi_2023_hourly.csv"]:
        syn_path = find_path(candidate)
        if os.path.exists(syn_path):
            break
    if not os.path.exists(syn_path):
        raise FileNotFoundError(
            "CSV not found. Place synthetic_ashesi_2022-09_to_2023-12_hourly_filled_pv.csv "
            "(preferred; gives January 2023 its full 3-month GP history) next to this script."
        )

    df = pd.read_csv(syn_path, parse_dates=["Datetime"]).sort_values("Datetime").set_index("Datetime")

    E_B_min, E_B_max = 300.0, 1200.0
    P_B_max = 500.0
    eta_ch, eta_dc = 0.95, 0.95
    eta_g = ETA_GEN_ELEC  # unified single generator efficiency (0.35)
    rho_deg, rho_curt, rho_shed = 0.02, 0.05, 1000.0
    E_B0 = 800.0

    nominal_day_sessions, _ = generate_ashesi_deterministic_day(N=24, dt=1.0, p_port=P_PORT_KW)
    p_diss_avg = compute_bus_avg_dissipation_from_daily_sessions(
        nominal_day_sessions, N=24, n_buses=NUM_BUSES, averaging_mode=P_DISS_AVG_MODE
    )


    outdir = "results_smpc_ev_buffer_alpha_sweep_paper_aligned"
    os.makedirs(outdir, exist_ok=True)

    monthly_rows = []
    monthly_ev_rows = []
    monthly_co2_rows = []
    initial_rows = []
    ts_rows = []
    compat_monthly_cost_rows = []
    compat_ev_violation_rows = []
    compat_ts_rows = []
    timing_rows = []
    timing_summary_rows = []

    for mon in MONTHS_TO_RUN:
        month_start = pd.Timestamp(mon + "-01 00:00:00")
        next_month_start = month_start + pd.offsets.MonthBegin(1)
        month_end = next_month_start - pd.Timedelta(hours=1)

        month_inputs = prepare_window_inputs(df, month_start, month_end, H_LOOK)
        T = len(month_inputs["index_window"])
        gp_model = fit_gp_three_month(df, month_start, seed=RNG_SEED, verbose=False)
        ev_month = build_ev_schedules_for_span(
            month_inputs["index_span"],
            deterministic_actual=USE_DETERMINISTIC_EV_SCENARIO,
            rng_seed=RNG_SEED
        )


        p_diss_plan_nominal_full = build_time_varying_bus_dissipation_profile(
            month_inputs["index_span"], ev_month["nominal_sessions_full"], NUM_BUSES, P_DISS_AVG_MODE
        )
        p_diss_actual_full = build_time_varying_bus_dissipation_profile(
            month_inputs["index_span"], ev_month["actual_sessions_full"], NUM_BUSES, P_DISS_AVG_MODE
        )

        common_E_bus0_init = None
        common_init_method = None
        common_init_lp_slack = None
        common_warm_seed = None
        common_init_ref_alpha = float(COMMON_INITIAL_BUS_ENERGY_REFERENCE_ALPHA)

        if HOLD_COMMON_INITIAL_BUS_ENERGY_ACROSS_ALPHA:
            common_E_bus0_init, common_init_method_raw, common_init_lp_slack, common_warm_seed = compute_initial_bus_energy_for_window(
                df=df,
                target_start=month_start,
                H_LOOK=H_LOOK,
                alpha_buffer=common_init_ref_alpha,
                p_diss_avg=p_diss_avg,
                nominal_day_sessions=nominal_day_sessions,
                E_B_min=E_B_min, E_B_max=E_B_max, P_B_max=P_B_max,
                eta_ch=eta_ch, eta_dc=eta_dc,
                eta_g=eta_g, rho_deg=rho_deg, rho_curt=rho_curt, rho_shed=rho_shed,
            )
            common_init_method = f"{common_init_method_raw}_COMMON_ACROSS_ALPHA_refalpha_{common_init_ref_alpha:.2f}"

        for alpha_buffer in ALPHA_SWEEP:
            if HOLD_COMMON_INITIAL_BUS_ENERGY_ACROSS_ALPHA:
                E_bus0_init = np.asarray(common_E_bus0_init, float).copy()
                init_method = common_init_method
                init_lp_slack = float(common_init_lp_slack)
                warm_seed = None if common_warm_seed is None else np.asarray(common_warm_seed, float).copy()
            else:
                E_bus0_init, init_method, init_lp_slack, warm_seed = compute_initial_bus_energy_for_window(
                    df=df,
                    target_start=month_start,
                    H_LOOK=H_LOOK,
                    alpha_buffer=alpha_buffer,
                    p_diss_avg=p_diss_avg,
                    nominal_day_sessions=nominal_day_sessions,
                    E_B_min=E_B_min, E_B_max=E_B_max, P_B_max=P_B_max,
                    eta_ch=eta_ch, eta_dc=eta_dc,
                    eta_g=eta_g, rho_deg=rho_deg, rho_curt=rho_curt, rho_shed=rho_shed,
                )

            initial_rows.append({
                "Month": mon,
                "Controller": CONTROLLER_NAME,
                "Alpha_Buffer": alpha_buffer,
                "Hold_Common_Initial_Bus_Energy_Across_Alpha": int(HOLD_COMMON_INITIAL_BUS_ENERGY_ACROSS_ALPHA),
                "Common_Init_Reference_Alpha": common_init_ref_alpha,
                "Init_Method": init_method,
                "Init_LP_Total_Slack_kWh": init_lp_slack,
                **({f"Bus_{b}_WarmSeed_kWh": float(warm_seed[b]) for b in range(NUM_BUSES)} if warm_seed is not None else {}),
                **{f"Bus_{b}_E0_kWh": float(E_bus0_init[b]) for b in range(NUM_BUSES)}
            })

            results = run_smpc_month_alpha(
                T=T, H=H_LOOK, dt=dt,
                df_full=df, index_span=month_inputs["index_span"],
                P_pv_max_full=month_inputs["P_pv_max_full"],
                P_grid_cap_full=month_inputs["P_grid_cap_full"],
                c_elec_full=month_inputs["c_elec_full"], c_gas_full=month_inputs["c_gas_full"],
                E_B_min=E_B_min, E_B_max=E_B_max, P_B_max=P_B_max, eta_ch=eta_ch, eta_dc=eta_dc,
                eta_g=eta_g, rho_deg=rho_deg, rho_curt=rho_curt, rho_shed=rho_shed,
                z_nom_full=ev_month["z_nom_full"], pmax_nom_full=ev_month["pmax_nom_full"],
                z_actual_full=ev_month["z_actual_full"], pmax_actual_full=ev_month["pmax_actual_full"],
                dep_req_actual_full=ev_month["dep_req_actual_full"],
                dep_event_count_full=ev_month["dep_event_count_full"],
                p_diss_plan_nominal_full=p_diss_plan_nominal_full,
                p_diss_actual_full=p_diss_actual_full,
                alpha_buffer=alpha_buffer,
                E_B0=E_B0,
                E_bus0_init=E_bus0_init,
                gp_model=gp_model,
                seed_outage=RNG_SEED,
                M_scen=M_SCEN,
                noise_scale=NOISE_SCALE,
                verbose=False,
            )

            node_df, dispatch_df, dep_metrics_actual = build_reporting_frames(
                mon=mon,
                alpha_buffer=alpha_buffer,
                index_month=month_inputs["index_window"],
                dfe=month_inputs["dfe"],
                P_pv_max_full=month_inputs["P_pv_max_full"],
                P_grid_cap_full=month_inputs["P_grid_cap_full"],
                c_elec_full=month_inputs["c_elec_full"],
                c_gas_full=month_inputs["c_gas_full"],
                z_nom_full=ev_month["z_nom_full"],
                pmax_nom_full=ev_month["pmax_nom_full"],
                z_actual_full=ev_month["z_actual_full"],
                pmax_actual_full=ev_month["pmax_actual_full"],
                dep_req_actual_full=ev_month["dep_req_actual_full"],
                dep_event_count_full=ev_month["dep_event_count_full"],
                p_diss_plan_nominal_full=p_diss_plan_nominal_full,
                p_diss_actual_full=p_diss_actual_full,
                results=results,
                dt=dt,
            )
            node_df["Controller"] = CONTROLLER_NAME
            dispatch_df["Controller"] = CONTROLLER_NAME

            grid_total = float(np.sum(results["grid_cost_k"]))
            diesel_total = float(np.sum(results["gas_cost_k"]))
            degr_total = float(np.sum(results["degr_cost_k"]))
            curt_total = float(np.sum(results["curt_cost_k"]))
            shed_total = float(np.sum(results["shed_cost_k"]))
            microgrid_total = grid_total + diesel_total + degr_total + curt_total + shed_total
            ev_slack_total = float(np.sum(results["ev_planned_slack_cost_k"]))
            ev_floor_clip_total = float(np.sum(results["ev_floor_clip_cost_k"]))
            total_with_ev_debug = microgrid_total + ev_slack_total + ev_floor_clip_total

            short_plan = np.maximum(EV_MIN_KWH - results["E_bus_plan"], 0.0)
            short_actual = np.maximum(EV_MIN_KWH - results["E_bus_real"], 0.0)
            ev_metrics_plan = compute_busnode_metrics(short_plan)
            ev_metrics_actual = compute_busnode_metrics(short_actual)
            sess_metrics_plan = compute_session_violation_metrics(
                short_plan, ev_month["nominal_sessions_full"], T_window=T, n_buses=NUM_BUSES
            )
            sess_metrics_actual = compute_session_violation_metrics(
                short_actual, ev_month["actual_sessions_full"], T_window=T, n_buses=NUM_BUSES
            )

            mco2 = diesel_co2_metrics(results["P_g"], dt_h=dt)
            monthly_co2_rows.append({"Month": mon, "Controller": CONTROLLER_NAME, "Alpha_Buffer": alpha_buffer, **mco2})

            min_actual_bus_energy = float(np.min(results["E_bus_real"]))
            avg_actual_bus_energy = float(np.mean(results["E_bus_real"]))
            floor_clip_total_kWh = float(np.sum(results["ev_floor_clip_kWh_k"]))

            # ---- per-step solver / computation times (R1 + R6) ----
            step_solve = np.asarray(results["solve_time_k"], float)
            step_wall = np.asarray(results["step_wall_k"], float)
            idx_win = pd.DatetimeIndex(month_inputs["index_window"])
            for _t in range(T):
                timing_rows.append({
                    "Month": mon,
                    "Controller": CONTROLLER_NAME,
                    "Alpha_Buffer": alpha_buffer,
                    "Step": _t,
                    "Datetime": idx_win[_t],
                    "QP_solve_time_s": float(step_solve[_t]),
                    "Step_wall_time_s": float(step_wall[_t]),
                    "M_scenarios": M_SCEN,
                    "Horizon_H": H_LOOK,
                })
            timing_summary_rows.append({
                "Month": mon,
                "Controller": CONTROLLER_NAME,
                "Alpha_Buffer": alpha_buffer,
                "steps": int(T),
                "QP_solve_mean_s": float(np.mean(step_solve)),
                "QP_solve_median_s": float(np.median(step_solve)),
                "QP_solve_p95_s": float(np.percentile(step_solve, 95)),
                "QP_solve_max_s": float(np.max(step_solve)),
                "Step_wall_mean_s": float(np.mean(step_wall)),
                "Step_wall_median_s": float(np.median(step_wall)),
                "Step_wall_p95_s": float(np.percentile(step_wall, 95)),
                "Step_wall_max_s": float(np.max(step_wall)),
            })

            monthly_rows.append({
                "Month": mon,
                "Controller": CONTROLLER_NAME,
                "Alpha_Buffer": alpha_buffer,
                "Hold_Common_Initial_Bus_Energy_Across_Alpha": int(HOLD_COMMON_INITIAL_BUS_ENERGY_ACROSS_ALPHA),
                "Common_Init_Reference_Alpha": common_init_ref_alpha,
                "Init_Method": init_method,
                "Init_LP_Total_Slack_kWh": init_lp_slack,
                "Grid": grid_total,
                "Diesel": diesel_total,
                "Batt_degr": degr_total,
                "PV_curt": curt_total,
                "Load_shed": shed_total,
                "Microgrid_Total": microgrid_total,
                "EV_slack_penalty": ev_slack_total,
                "EV_floor_clip_penalty_debug": ev_floor_clip_total,
                "Total_with_EV_penalties_debug": total_with_ev_debug,
                "EV_nodes": ev_metrics_plan["nodes_total"],
                "EV_nodes_viol": ev_metrics_plan["nodes_viol"],
                "EV_viol_frac": ev_metrics_plan["frac_viol"],
                "EV_avg_short_kWh": ev_metrics_plan["avg_short_kWh"],
                "EV_max_short_kWh": ev_metrics_plan["max_short_kWh"],
                "EV_total_short_kWh": ev_metrics_plan["total_short_kWh"],
                "EV_actual_nodes_viol": ev_metrics_actual["nodes_viol"],
                "EV_actual_viol_frac": ev_metrics_actual["frac_viol"],
                "EV_actual_avg_short_kWh": ev_metrics_actual["avg_short_kWh"],
                "EV_actual_max_short_kWh": ev_metrics_actual["max_short_kWh"],
                "EV_actual_total_short_kWh": ev_metrics_actual["total_short_kWh"],
                "Departure_events_total": dep_metrics_actual["events_total"],
                "Departure_events_viol": dep_metrics_actual["events_viol"],
                "Departure_viol_frac": dep_metrics_actual["viol_frac"],
                "Departure_avg_short_kWh": dep_metrics_actual["avg_short_kWh"],
                "Departure_max_short_kWh": dep_metrics_actual["max_short_kWh"],
                "Departure_total_short_kWh": dep_metrics_actual["total_short_kWh"],
                "Floor_Clip_Total_kWh": floor_clip_total_kWh,
                "Floor_Clip_Nonzero_Flag": int(floor_clip_total_kWh > FLOOR_CLIP_WARN_TOL_KWH),
                "Min_actual_bus_energy_kWh": min_actual_bus_energy,
                "Avg_actual_bus_energy_kWh": avg_actual_bus_energy,
            })

            monthly_ev_rows.append({
                "Month": mon,
                "Controller": CONTROLLER_NAME,
                "Alpha_Buffer": alpha_buffer,
                "Hold_Common_Initial_Bus_Energy_Across_Alpha": int(HOLD_COMMON_INITIAL_BUS_ENERGY_ACROSS_ALPHA),
                "Common_Init_Reference_Alpha": common_init_ref_alpha,
                "Init_Method": init_method,
                "Init_LP_Total_Slack_kWh": init_lp_slack,
                "P_diss_total_kW": float(np.mean(p_diss_actual_full[:, :T].sum(axis=0))),
                "P_diss_controller_nominal_total_kW": float(np.mean(p_diss_plan_nominal_full[:, :T].sum(axis=0))),
                "floor_clip_total_kWh": floor_clip_total_kWh,
                "planned_nodes_total": ev_metrics_plan["nodes_total"],
                "planned_nodes_viol": ev_metrics_plan["nodes_viol"],
                "planned_viol_frac": ev_metrics_plan["frac_viol"],
                "planned_lower_bound_violation_rate_bus_timestep": ev_metrics_plan["frac_viol"],
                "planned_avg_short_kWh": ev_metrics_plan["avg_short_kWh"],
                "planned_max_short_kWh": ev_metrics_plan["max_short_kWh"],
                "planned_total_short_kWh": ev_metrics_plan["total_short_kWh"],
                "planned_sessions_total": sess_metrics_plan["sessions_total"],
                "planned_sessions_associated_with_violation": sess_metrics_plan["sessions_viol"],
                "planned_session_violation_rate": sess_metrics_plan["viol_rate"],
                "actual_nodes_total": ev_metrics_actual["nodes_total"],
                "actual_nodes_viol": ev_metrics_actual["nodes_viol"],
                "actual_viol_frac": ev_metrics_actual["frac_viol"],
                "actual_lower_bound_violation_rate_bus_timestep": ev_metrics_actual["frac_viol"],
                "actual_avg_short_kWh": ev_metrics_actual["avg_short_kWh"],
                "actual_max_short_kWh": ev_metrics_actual["max_short_kWh"],
                "actual_total_short_kWh": ev_metrics_actual["total_short_kWh"],
                "actual_sessions_total": sess_metrics_actual["sessions_total"],
                "actual_sessions_associated_with_violation": sess_metrics_actual["sessions_viol"],
                "actual_session_violation_rate": sess_metrics_actual["viol_rate"],
                "departure_events_total": dep_metrics_actual["events_total"],
                "departure_events_viol": dep_metrics_actual["events_viol"],
                "departure_viol_frac": dep_metrics_actual["viol_frac"],
                "departure_violation_rate": dep_metrics_actual["viol_frac"],
                "departure_avg_short_kWh": dep_metrics_actual["avg_short_kWh"],
                "departure_max_short_kWh": dep_metrics_actual["max_short_kWh"],
                "departure_total_short_kWh": dep_metrics_actual["total_short_kWh"],
                "min_actual_bus_energy_kWh": min_actual_bus_energy,
                "avg_actual_bus_energy_kWh": avg_actual_bus_energy,
            })

            compat_monthly_cost_rows.append({
                "Month": mon,
                "Controller": CONTROLLER_NAME,
                "Alpha_Buffer": alpha_buffer,
                "Grid_GHS": grid_total,
                "Diesel_GHS": diesel_total,
                "Batt_Degr_GHS": degr_total,
                "PV_Curt_GHS": curt_total,
                "Load_Shed_GHS": shed_total,
                "EV_Slack_GHS": ev_slack_total,
                "EV_Floor_Clip_GHS_Debug": ev_floor_clip_total,
                "Microgrid_Total_GHS": microgrid_total,
                "Total_with_EV_GHS_Debug": total_with_ev_debug,
            })

            compat_ev_violation_rows.append({
                "Month": mon,
                "Controller": CONTROLLER_NAME,
                "Alpha_Buffer": alpha_buffer,
                "target_type": "lower_bound_bus_node_planned",
                "Nodes_or_Events": ev_metrics_plan["nodes_total"],
                "Viol_count": ev_metrics_plan["nodes_viol"],
                "Viol_frac": ev_metrics_plan["frac_viol"],
                "Avg_short_kWh": ev_metrics_plan["avg_short_kWh"],
                "Max_short_kWh": ev_metrics_plan["max_short_kWh"],
                "Total_short_kWh": ev_metrics_plan["total_short_kWh"],
            })
            compat_ev_violation_rows.append({
                "Month": mon,
                "Controller": CONTROLLER_NAME,
                "Alpha_Buffer": alpha_buffer,
                "target_type": "lower_bound_bus_node_actual",
                "Nodes_or_Events": ev_metrics_actual["nodes_total"],
                "Viol_count": ev_metrics_actual["nodes_viol"],
                "Viol_frac": ev_metrics_actual["frac_viol"],
                "Avg_short_kWh": ev_metrics_actual["avg_short_kWh"],
                "Max_short_kWh": ev_metrics_actual["max_short_kWh"],
                "Total_short_kWh": ev_metrics_actual["total_short_kWh"],
            })
            compat_ev_violation_rows.append({
                "Month": mon,
                "Controller": CONTROLLER_NAME,
                "Alpha_Buffer": alpha_buffer,
                "target_type": "departure_energy_actual",
                "Nodes_or_Events": dep_metrics_actual["events_total"],
                "Viol_count": dep_metrics_actual["events_viol"],
                "Viol_frac": dep_metrics_actual["viol_frac"],
                "Avg_short_kWh": dep_metrics_actual["avg_short_kWh"],
                "Max_short_kWh": dep_metrics_actual["max_short_kWh"],
                "Total_short_kWh": dep_metrics_actual["total_short_kWh"],
            })

            ts_rows.append(dispatch_df)
            compat_ts_rows.append(dispatch_df.copy())


    monthly_df = pd.DataFrame(monthly_rows)
    monthly_wide_path = os.path.join(outdir, "smpc_alpha_sweep_monthly_wide.csv")
    monthly_df.to_csv(monthly_wide_path, index=False)

    co2_df = pd.DataFrame(monthly_co2_rows)
    co2_path = os.path.join(outdir, "smpc_alpha_sweep_monthly_diesel_co2.csv")
    co2_df.to_csv(co2_path, index=False)

    ev_metrics_df = pd.DataFrame(monthly_ev_rows)
    ev_metrics_path = os.path.join(outdir, "smpc_alpha_sweep_monthly_ev_metrics.csv")
    ev_metrics_df.to_csv(ev_metrics_path, index=False)

    init_df = pd.DataFrame(initial_rows)
    init_path = os.path.join(outdir, "smpc_alpha_sweep_initial_bus_energy.csv")
    init_df.to_csv(init_path, index=False)

    ts_df = pd.concat(ts_rows, ignore_index=True)
    ts_df["Datetime"] = pd.to_datetime(ts_df["Datetime"])
    ts_df = ts_df.sort_values(["Alpha_Buffer", "Month", "Datetime"]).reset_index(drop=True)
    ts_path = os.path.join(outdir, "smpc_alpha_sweep_timeseries.csv")
    ts_df.to_csv(ts_path, index=False)

    compat_monthly_cost_df = pd.DataFrame(compat_monthly_cost_rows)
    compat_monthly_cost_path = os.path.join(outdir, "monthly_cost_breakdown.csv")
    compat_monthly_cost_df.to_csv(compat_monthly_cost_path, index=False)

    compat_ev_violation_df = pd.DataFrame(compat_ev_violation_rows)
    compat_ev_violation_path = os.path.join(outdir, "monthly_ev_violation_summary.csv")
    compat_ev_violation_df.to_csv(compat_ev_violation_path, index=False)

    compat_ts_df = pd.concat(compat_ts_rows, ignore_index=True)
    compat_ts_df["Datetime"] = pd.to_datetime(compat_ts_df["Datetime"])
    compat_ts_df = compat_ts_df.sort_values(["Month", "Datetime"]).reset_index(drop=True)
    compat_ts_path = os.path.join(outdir, "timeseries_dispatch.csv")
    compat_ts_df.to_csv(compat_ts_path, index=False)

    timing_df = pd.DataFrame(timing_rows)
    timing_df["Datetime"] = pd.to_datetime(timing_df["Datetime"])
    timing_df = timing_df.sort_values(["Alpha_Buffer", "Month", "Datetime"]).reset_index(drop=True)
    timing_path = os.path.join(outdir, "smpc_alpha_sweep_solve_times.csv")
    timing_df.to_csv(timing_path, index=False)

    timing_summary_df = pd.DataFrame(timing_summary_rows)
    timing_summary_path = os.path.join(outdir, "smpc_alpha_sweep_solve_time_summary.csv")
    timing_summary_df.to_csv(timing_summary_path, index=False)


